# QIMED — Pipeline quântico com validação nested temporal e SHAP

Este notebook é uma versão metodologicamente revisada de `QIMED_testes-quantum.ipynb`.
O notebook original não é modificado.

Objetivos principais:

- usar uma amostra estratificada e reprodutível de 1.000 registros, reordenada no tempo;
- usar os mesmos 10 outer folds nos cenários clássico, quantum-only e híbrido;
- executar 3 inner folds temporais com tuning por Average Precision;
- ajustar preprocessing, seleção SHAP, PQFM e classificador somente no treino de cada split;
- avaliar três arquiteturas XYZ independentes nas variantes `diagonal`, `cross-only` e `full`, além de uma camada Heisenberg independente.


In [1]:
import json
import os
import pickle
import time
import warnings
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from scipy.stats import t as student_t
from scipy.stats import wilcoxon

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedShuffleSplit, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pqfmlib import HeisenbergProjectiveQFM, XYZProjectiveQFM

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_SAMPLES = 1_000
OUTER_SPLITS = 10
INNER_SPLITS = 3
SCORING = "average_precision"
SCENARIOS = ("classica", "quantum_only", "hibrida")
METRICS = ("ROC-AUC", "PR-AUC", "Accuracy", "Precision", "Recall", "F1", "Specificity")


/home/rafael/projects/pqfmlib-api/.venv_gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dados e amostragem estratificada

A estratificação é feita pelo target com semente fixa. Depois da seleção, os 1.000 registros
são novamente ordenados por `period_start`. Assim, a amostra conserva a proporção de classes
sem perder a ordem exigida pelo `TimeSeriesSplit`.


In [2]:
DATA_PATH = Path("./data/dataset_modelo_readmissao.parquet")
TARGET_COL = "readmitted_30d"

FEATURE_COLS = [
    "age_at_enc", "gender", "race", "deceased", "marital_status",
    "class_code", "enc_type_grp", "has_reason",
    "n_enc_total", "n_enc_30d", "n_enc_90d", "n_enc_365d",
    "days_since_last", "had_emer_90d", "had_imp_90d",
    "month", "quarter", "day_of_week", "is_weekend",
    "has_renal_disease", "has_diabetes", "has_hypertension", "has_mental_health",
    "n_conditions_total", "n_conditions_active",
    "last_hba1c", "last_egfr", "last_systolic_bp", "last_diastolic_bp", "last_bmi",
    "n_labs_90d", "n_vitals_90d",
    "n_procedures_90d", "n_procedures_365d", "had_surgical_90d", "had_dialysis_90d",
]

NUM_COLS = [
    "age_at_enc", "n_enc_total", "n_enc_30d", "n_enc_90d", "n_enc_365d",
    "days_since_last", "n_conditions_total", "n_conditions_active",
    "n_procedures_90d", "n_procedures_365d", "n_labs_90d", "n_vitals_90d",
]
LAB_COLS = ["last_hba1c", "last_egfr", "last_systolic_bp", "last_diastolic_bp", "last_bmi"]
CAT_COLS = [
    "gender", "race", "marital_status", "class_code", "enc_type_grp",
    "month", "day_of_week", "quarter",
]
BIN_COLS = [
    "deceased", "has_reason", "had_emer_90d", "had_imp_90d",
    "has_renal_disease", "has_diabetes", "has_hypertension", "has_mental_health",
    "had_surgical_90d", "had_dialysis_90d", "is_weekend",
]


def stratified_temporal_sample(df, target_col=TARGET_COL, n_samples=N_SAMPLES, random_state=RANDOM_STATE):
    if len(df) < n_samples:
        raise ValueError(f"O dataset possui {len(df)} linhas; são necessárias pelo menos {n_samples}.")
    ordered = df.sort_values("period_start", kind="mergesort").reset_index().rename(
        columns={"index": "source_index"}
    )
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=n_samples, random_state=random_state)
    sample_idx, _ = next(splitter.split(ordered, ordered[target_col]))
    sampled = ordered.iloc[sample_idx].sort_values("period_start", kind="mergesort").reset_index(drop=True)
    return sampled


df = pd.read_parquet(DATA_PATH)
sampled_df = stratified_temporal_sample(df)
X_sample = sampled_df[FEATURE_COLS].copy()
y_sample = sampled_df[TARGET_COL].astype(int).copy()

outer_cv = TimeSeriesSplit(n_splits=OUTER_SPLITS)
OUTER_FOLDS = [(train.copy(), test.copy()) for train, test in outer_cv.split(X_sample)]

print(f"Amostra: {X_sample.shape}")
print(f"Distribuição original: {df[TARGET_COL].value_counts(normalize=True).sort_index().round(4).to_dict()}")
print(f"Distribuição amostral: {y_sample.value_counts(normalize=True).sort_index().round(4).to_dict()}")
print(f"Período: {sampled_df['period_start'].min()} → {sampled_df['period_start'].max()}")
print(f"Outer folds compartilhados: {len(OUTER_FOLDS)}")


Amostra: (1000, 36)
Distribuição original: {0: 0.3195, 1: 0.6805}
Distribuição amostral: {0: 0.32, 1: 0.68}
Período: 1952-01-18 02:54:36+00:00 → 2023-03-17 02:54:36+00:00
Outer folds compartilhados: 10


In [3]:
def describe_temporal_folds(y, folds=OUTER_FOLDS):
    rows = []
    for fold, (train_idx, test_idx) in enumerate(folds, 1):
        rows.append({
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "train_positive_rate": y.iloc[train_idx].mean(),
            "test_positive_rate": y.iloc[test_idx].mean(),
            "test_start": sampled_df.iloc[test_idx]["period_start"].min(),
            "test_end": sampled_df.iloc[test_idx]["period_start"].max(),
        })
    return pd.DataFrame(rows)


outer_fold_description = describe_temporal_folds(y_sample)
outer_fold_description


,fold,n_train,n_test,train_positive_rate,test_positive_rate,test_start,test_end
0,1,100,90,0.670000,0.911111,1983-03-15 04:33:36+00:00,1988-01-22 02:54:36+00:00
1,2,190,90,0.784211,0.655556,1988-04-14 09:29:36+00:00,2002-12-17 15:17:48+00:00
2,3,280,90,0.742857,0.744444,2002-12-23 21:35:48+00:00,2011-09-23 02:11:33+00:00
3,4,370,90,0.743243,0.711111,2011-09-24 11:55:03+00:00,2014-03-17 15:35:54+00:00
4,5,460,90,0.736957,0.566667,2014-03-17 22:34:00+00:00,2015-11-06 08:10:05+00:00
5,6,550,90,0.709091,0.666667,2015-11-12 14:39:36+00:00,2017-05-13 18:34:02+00:00
6,7,640,90,0.703125,0.522222,2017-05-23 11:13:12+00:00,2019-01-24 15:05:43+00:00
7,8,730,90,0.680822,0.588889,2019-01-27 03:58:16+00:00,2020-05-09 07:16:28+00:00
8,9,820,90,0.670732,0.766667,2020-05-15 12:45:37+00:00,2021-09-24 02:54:36+00:00
9,10,910,90,0.680220,0.677778,2021-09-28 14:15:28+00:00,2023-03-17 02:54:36+00:00


## 2. Esquema da validação nested temporal — 10 outer × 3 inner

```text
Amostra estratificada (1.000), novamente ordenada por period_start
                            │
                 TimeSeriesSplit outer (10)
                            │
          ┌─────────────────┴──────────────────┐
          │ outer_train                        │ outer_test (intocado)
          │                                    │
          │ TimeSeriesSplit inner (3)           │
          │                                    │
          │  inner_train:                      │
          │    fit preprocessing               │
          │             ↓                      │
          │    fit seleção SHAP                │
          │             ↓                      │
          │    fit PQFM (*)                    │
          │             ↓                      │
          │    fit classificador               │
          │                                    │
          │  inner_validation:                 │
          │    transform → transform →         │
          │    transform PQFM (*) → predict    │
          │                                    │
          └──────── escolher hiperparâmetros ──┘
                            │
              refit de toda a cadeia usando
                     todo o outer_train
                            │
                 transform/predict outer_test
```

`(*)` A PQFM é usada somente nos cenários quantum-only e híbrido. O cenário clássico segue
diretamente das features selecionadas por SHAP para o classificador.


## 3. Funções reutilizáveis: preprocessing e seleção SHAP

O seletor SHAP ajusta uma Random Forest somente no conjunto de treino recebido e ranqueia
features pela média do valor SHAP absoluto. Nenhuma estatística do validation/test participa.


In [4]:
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), NUM_COLS),
            ("labs", Pipeline([
                ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                ("scaler", StandardScaler()),
            ]), LAB_COLS),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), CAT_COLS),
            ("bin", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ]), BIN_COLS),
        ]
    )


def _as_dense_frame(values, columns, index=None):
    if hasattr(values, "toarray"):
        values = values.toarray()
    return pd.DataFrame(np.asarray(values, dtype=float), columns=list(columns), index=index).reset_index(drop=True)


def fit_preprocessing(X_train):
    preprocessor = build_preprocessor()
    transformed = preprocessor.fit_transform(X_train)
    columns = preprocessor.get_feature_names_out()
    X_frame = _as_dense_frame(transformed, columns)

    # Escala também one-hot/binárias para entregar uma matriz homogênea à PQFM.
    final_scaler = StandardScaler()
    X_scaled = pd.DataFrame(final_scaler.fit_transform(X_frame), columns=columns)
    return {"preprocessor": preprocessor, "final_scaler": final_scaler, "columns": list(columns)}, X_scaled


def transform_preprocessing(fitted, X):
    transformed = fitted["preprocessor"].transform(X)
    X_frame = _as_dense_frame(transformed, fitted["columns"])
    return pd.DataFrame(fitted["final_scaler"].transform(X_frame), columns=fitted["columns"])


def _positive_class_shap_matrix(shap_output, n_samples, n_features):
    values = getattr(shap_output, "values", shap_output)
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)

    if values.ndim == 3:
        if values.shape[:2] == (n_samples, n_features):
            values = values[:, :, -1]
        elif values.shape[1:] == (n_samples, n_features):
            values = values[-1]
        else:
            raise ValueError(f"Formato SHAP 3D não reconhecido: {values.shape}")
    if values.shape != (n_samples, n_features):
        raise ValueError(f"Formato SHAP inesperado: {values.shape}; esperado {(n_samples, n_features)}")
    return values


def fit_shap_selector(X_train, y_train, n_features, random_state=RANDOM_STATE):
    if n_features > X_train.shape[1]:
        raise ValueError(f"Foram solicitadas {n_features} features, mas há apenas {X_train.shape[1]}.")
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    shap_output = shap.TreeExplainer(model)(X_train, check_additivity=False)
    shap_matrix = _positive_class_shap_matrix(shap_output, len(X_train), X_train.shape[1])
    importance = pd.Series(np.abs(shap_matrix).mean(axis=0), index=X_train.columns)
    selected = importance.sort_values(ascending=False, kind="mergesort").head(n_features).index.tolist()
    return {"model": model, "selected_features": selected, "importance": importance.sort_values(ascending=False)}


def transform_shap_selection(selector, X):
    return X.loc[:, selector["selected_features"]].copy().reset_index(drop=True)


def fit_feature_stack(X_train, y_train, n_features):
    preprocessing, X_preprocessed = fit_preprocessing(X_train)
    selector = fit_shap_selector(X_preprocessed, y_train.reset_index(drop=True), n_features)
    X_selected = transform_shap_selection(selector, X_preprocessed)
    return {"preprocessing": preprocessing, "selector": selector}, X_selected


def transform_feature_stack(fitted, X):
    X_preprocessed = transform_preprocessing(fitted["preprocessing"], X)
    return transform_shap_selection(fitted["selector"], X_preprocessed)


## 4. Credencial e variantes PQFM

Não há função nem dicionário de arquitetura que construa uma PQFM. Cada seção executável
declara seus próprios parâmetros e instancia `XYZProjectiveQFM` explicitamente.

Em cada célula quântica, `VARIANTS_TO_RUN` permite executar uma, duas ou as três variantes.


In [5]:
QISKIT_CHANNEL = "ibm_cloud"
QPU_TARGET = "ibm_marrakesh"
SHOTS = 4096


def ensure_ibm_token():
    if not os.environ.get("QISKIT_IBM_TOKEN"):
        token = getpass("IBM Cloud API token: ").strip()
        if not token:
            raise ValueError("Informe um token IBM Cloud para executar com ideal=False.")
        os.environ["QISKIT_IBM_TOKEN"] = token
        del token
    print(f"Credencial disponível somente no ambiente da sessão. Backend: {QPU_TARGET}")


# Registro flexível: cada célula salva seu resultado usando RUN_LABEL como chave.
if "EXPERIMENTS" not in globals():
    EXPERIMENTS = {}

# O backend é consultado uma única vez por sessão e reutilizado por todas as PQFMs.
from pqfmlib.core.backend import load_ibm_backend
from pqfmlib.utils.io import save_json

FIXED_LAYOUT_DIR = Path("outputs/fixed_phys_nodes")
FIXED_LAYOUT_DIR.mkdir(parents=True, exist_ok=True)
FIXED_PHYS_NODES_12Q_FILE = FIXED_LAYOUT_DIR / f"{QPU_TARGET}_12q.json"
FIXED_PHYS_NODES_18Q_FILE = FIXED_LAYOUT_DIR / f"{QPU_TARGET}_18q.json"

ensure_ibm_token()
if "SHARED_REAL_BACKEND" not in globals():
    print(f"Carregando metadados de {QPU_TARGET} uma única vez...")
    SHARED_REAL_BACKEND, SHARED_IBM_SERVICE = load_ibm_backend(
        QPU_TARGET,
        channel=QISKIT_CHANNEL,
        token=os.environ["QISKIT_IBM_TOKEN"],
    )
    print(f"Backend {QPU_TARGET} carregado e mantido em memória.")
else:
    print(f"Backend {QPU_TARGET} já está em memória; nenhuma nova conexão foi feita.")

for layout_file in (FIXED_PHYS_NODES_12Q_FILE, FIXED_PHYS_NODES_18Q_FILE):
    status = "disponível" if layout_file.exists() else "será criado no primeiro fit"
    print(f"Layout {layout_file.name}: {status}")


IBM Cloud API token:  ········


Credencial disponível somente no ambiente da sessão. Backend: ibm_marrakesh
Carregando metadados de ibm_marrakesh uma única vez...


qiskit_runtime_service._resolve_cloud_instances:WARNING:2026-08-25 23:55:13,192: Default instance not set. Searching all available instances.


Backend ibm_marrakesh carregado e mantido em memória.
Layout ibm_marrakesh_12q.json: disponível
Layout ibm_marrakesh_18q.json: disponível


## 5. Cache do pipeline clássico

O clássico é preparado uma única vez para 12 features e uma única vez para 36 features.
Cada cache conserva, por outer fold:

- as matrizes clássicas de cada inner train/validation após preprocessing e SHAP;
- o tuning clássico por Average Precision;
- o refit clássico no outer train e sua avaliação no outer test.

As células PQFM apenas leem esses dados. As arquiteturas 12q×3f e 18q×2f compartilham o
mesmo cache de 36 features.


In [6]:
CLASSIFIER_GRID = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
}


def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_proba = np.asarray(y_proba)
    roc_auc = roc_auc_score(y_true, y_proba) if np.unique(y_true).size == 2 else np.nan
    pr_auc = average_precision_score(y_true, y_proba) if np.any(y_true == 1) else np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
    }


def candidate_key(params):
    return json.dumps(params, sort_keys=True)


def fit_and_score_classifier(X_train, y_train, X_validation, y_validation, params):
    classifier = GradientBoostingClassifier(random_state=RANDOM_STATE, **params)
    classifier.fit(X_train, y_train)
    probabilities = classifier.predict_proba(X_validation)[:, 1]
    return average_precision_score(y_validation, probabilities)


def prepare_classical_cache(n_shap_features, X=X_sample, y=y_sample, outer_folds=OUTER_FOLDS):
    candidates = list(ParameterGrid(CLASSIFIER_GRID))
    cached_folds = []

    for outer_fold, (train_idx, test_idx) in enumerate(outer_folds, 1):
        print(f"[clássico/{n_shap_features} features] outer fold {outer_fold}/{len(outer_folds)}")
        X_outer_train = X.iloc[train_idx].copy()
        X_outer_test = X.iloc[test_idx].copy()
        y_outer_train = y.iloc[train_idx].reset_index(drop=True)
        y_outer_test = y.iloc[test_idx].reset_index(drop=True)

        inner_data = []
        classic_scores = {candidate_key(params): [] for params in candidates}
        inner_cv = TimeSeriesSplit(n_splits=INNER_SPLITS)

        for inner_fold, (inner_train_idx, inner_validation_idx) in enumerate(
            inner_cv.split(X_outer_train), 1
        ):
            X_inner_train_raw = X_outer_train.iloc[inner_train_idx].copy()
            X_inner_validation_raw = X_outer_train.iloc[inner_validation_idx].copy()
            y_inner_train = y_outer_train.iloc[inner_train_idx].reset_index(drop=True)
            y_inner_validation = y_outer_train.iloc[inner_validation_idx].reset_index(drop=True)

            fitted_stack, X_inner_train = fit_feature_stack(
                X_inner_train_raw, y_inner_train, n_shap_features
            )
            X_inner_validation = transform_feature_stack(fitted_stack, X_inner_validation_raw)

            for params in candidates:
                classic_scores[candidate_key(params)].append(
                    fit_and_score_classifier(
                        X_inner_train,
                        y_inner_train,
                        X_inner_validation,
                        y_inner_validation,
                        params,
                    )
                )

            inner_data.append({
                "inner_fold": inner_fold,
                "X_train": X_inner_train,
                "X_validation": X_inner_validation,
                "y_train": y_inner_train,
                "y_validation": y_inner_validation,
                "selected_features": fitted_stack["selector"]["selected_features"],
            })

        mean_scores = {key: float(np.mean(values)) for key, values in classic_scores.items()}
        best_classic_key = max(mean_scores, key=mean_scores.get)
        best_classic_params = json.loads(best_classic_key)

        fitted_outer_stack, X_outer_train_classic = fit_feature_stack(
            X_outer_train, y_outer_train, n_shap_features
        )
        X_outer_test_classic = transform_feature_stack(fitted_outer_stack, X_outer_test)

        classic_classifier = GradientBoostingClassifier(
            random_state=RANDOM_STATE, **best_classic_params
        )
        classic_classifier.fit(X_outer_train_classic, y_outer_train)
        classic_prediction = classic_classifier.predict(X_outer_test_classic)
        classic_probability = classic_classifier.predict_proba(X_outer_test_classic)[:, 1]

        cached_folds.append({
            "outer_fold": outer_fold,
            "train_idx": train_idx.copy(),
            "test_idx": test_idx.copy(),
            "inner_folds": inner_data,
            "X_outer_train": X_outer_train_classic,
            "X_outer_test": X_outer_test_classic,
            "y_outer_train": y_outer_train,
            "y_outer_test": y_outer_test,
            "selected_features": fitted_outer_stack["selector"]["selected_features"],
            "best_classic_params": best_classic_params,
            "classic_metrics": compute_metrics(
                y_outer_test, classic_prediction, classic_probability
            ),
            "classic_tuning_scores": classic_scores,
        })

    return {"n_shap_features": n_shap_features, "folds": cached_folds}


### 5.1 Cache clássico — 12 features

Execute esta célula antes da seção 12q × 1f. O `if` evita recalcular o cache quando a célula
for executada novamente na mesma sessão.


In [7]:
if "CLASSIC_CACHE_12" not in globals():
    CLASSIC_CACHE_12 = prepare_classical_cache(n_shap_features=12)
else:
    print("CLASSIC_CACHE_12 já existe; reutilizando dados clássicos.")


[clássico/12 features] outer fold 1/10
[clássico/12 features] outer fold 2/10
[clássico/12 features] outer fold 3/10
[clássico/12 features] outer fold 4/10
[clássico/12 features] outer fold 5/10
[clássico/12 features] outer fold 6/10
[clássico/12 features] outer fold 7/10
[clássico/12 features] outer fold 8/10
[clássico/12 features] outer fold 9/10
[clássico/12 features] outer fold 10/10


### 5.2 Cache clássico — 36 features

Execute esta célula antes das seções 12q × 3f ou 18q × 2f. O mesmo cache é reutilizado pelas
duas arquiteturas.


In [8]:
if "CLASSIC_CACHE_36" not in globals():
    CLASSIC_CACHE_36 = prepare_classical_cache(n_shap_features=36)
else:
    print("CLASSIC_CACHE_36 já existe; reutilizando dados clássicos.")


[clássico/36 features] outer fold 1/10
[clássico/36 features] outer fold 2/10
[clássico/36 features] outer fold 3/10
[clássico/36 features] outer fold 4/10
[clássico/36 features] outer fold 5/10
[clássico/36 features] outer fold 6/10
[clássico/36 features] outer fold 7/10
[clássico/36 features] outer fold 8/10
[clássico/36 features] outer fold 9/10
[clássico/36 features] outer fold 10/10


## 6. Funções de apresentação e comparação

Estas funções não criam nem executam PQFMs; apenas reorganizam resultados já calculados.


In [9]:
def results_long(fold_results):
    id_columns = [
        "config", "variant", "scenario", "outer_fold", "n_train", "n_test",
        "test_start_position", "test_end_position", "best_params",
    ]
    return fold_results.melt(
        id_vars=id_columns,
        value_vars=list(METRICS),
        var_name="metric",
        value_name="value",
    )


def summarize_results(fold_results, confidence=0.95):
    long_df = results_long(fold_results)
    rows = []
    for keys, group in long_df.groupby(["config", "variant", "scenario", "metric"], sort=False):
        values = group["value"].dropna().to_numpy(dtype=float)
        n = len(values)
        mean = np.mean(values) if n else np.nan
        std = np.std(values, ddof=1) if n > 1 else np.nan
        critical = student_t.ppf((1 + confidence) / 2, df=n - 1) if n > 1 else np.nan
        margin = critical * std / np.sqrt(n) if n > 1 else np.nan
        rows.append({
            "config": keys[0], "variant": keys[1], "scenario": keys[2], "metric": keys[3],
            "n_folds": n, "mean": mean, "std": std,
            "ci95_low": mean - margin if n > 1 else np.nan,
            "ci95_high": mean + margin if n > 1 else np.nan,
        })
    return pd.DataFrame(rows)


def paired_wilcoxon_table(fold_results, comparisons=None):
    comparisons = comparisons or [
        ("classica", "quantum_only"),
        ("classica", "hibrida"),
    ]
    long_df = results_long(fold_results)
    rows = []
    for (config, variant, metric), group in long_df.groupby(
        ["config", "variant", "metric"], sort=False
    ):
        paired = group.pivot(index="outer_fold", columns="scenario", values="value")
        for baseline, challenger in comparisons:
            pair = paired[[baseline, challenger]].dropna()
            delta = pair[challenger] - pair[baseline]
            if len(pair) < 2:
                statistic, p_value = np.nan, np.nan
            elif np.allclose(delta, 0):
                statistic, p_value = 0.0, 1.0
            else:
                statistic, p_value = wilcoxon(
                    pair[challenger], pair[baseline], alternative="two-sided"
                )
            rows.append({
                "config": config,
                "variant": variant,
                "metric": metric,
                "baseline": baseline,
                "challenger": challenger,
                "n_pairs": len(pair),
                "mean_delta": delta.mean(),
                "wilcoxon_statistic": statistic,
                "p_value": p_value,
            })
    return pd.DataFrame(rows)


def analise_estatistica(
    fold_results,
    titulo="ANÁLISE ESTATÍSTICA",
    n_bootstrap=2_000,
    random_state=RANDOM_STATE,
):
    comparacoes = [
        ("classica", "quantum_only", "Clássico vs Quantum-Only"),
        ("classica", "hibrida", "Clássico vs Híbrido"),
    ]
    rng = np.random.default_rng(random_state)
    rows = []

    for (config, variant), group in fold_results.groupby(["config", "variant"], sort=False):
        for metrica in METRICS:
            paired = group.pivot(index="outer_fold", columns="scenario", values=metrica)
            for rep_base, rep_caso, label in comparacoes:
                pair = paired[[rep_base, rep_caso]].dropna()
                if len(pair) < 2:
                    continue

                base = pair[rep_base].to_numpy(dtype=float)
                caso = pair[rep_caso].to_numpy(dtype=float)
                delta = caso - base
                delta_medio = float(np.mean(delta))

                if np.allclose(delta, 0):
                    statistic, p_val = 0.0, 1.0
                else:
                    try:
                        statistic, p_val = wilcoxon(caso, base, alternative="two-sided")
                    except ValueError:
                        statistic, p_val = np.nan, np.nan

                bootstrap_indices = rng.integers(0, len(delta), size=(n_bootstrap, len(delta)))
                bootstrap_means = delta[bootstrap_indices].mean(axis=1)
                ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])

                significativo = bool(p_val < 0.05) if not np.isnan(p_val) else False
                if np.isclose(delta_medio, 0):
                    status = "Sem alteração"
                elif significativo and delta_medio > 0:
                    status = "✅ MELHORA SIGNIFICATIVA"
                elif significativo and delta_medio < 0:
                    status = "⚠️ PIORA SIGNIFICATIVA"
                elif delta_medio > 0:
                    status = "↑ Melhora (não sig.)"
                else:
                    status = "↓ Piora (não sig.)"

                rows.append({
                    "Config": config,
                    "Variante": variant,
                    "Comparação": label,
                    "Métrica": metrica,
                    "Folds": len(pair),
                    "Média Base": float(np.mean(base)),
                    "Média Caso": float(np.mean(caso)),
                    "Δ Médio": delta_medio,
                    "IC95 low": float(ci_low),
                    "IC95 high": float(ci_high),
                    "Estatística W": float(statistic) if not np.isnan(statistic) else np.nan,
                    "p-valor": float(p_val) if not np.isnan(p_val) else np.nan,
                    "Status": status,
                })

    df = pd.DataFrame(rows)

    print(f"\n{'=' * 88}")
    print(f"  {titulo}")
    print(f"{'=' * 88}")

    if df.empty:
        print("Nenhuma comparação pareada disponível.")
        return df

    melhorias = df[df["Status"] == "✅ MELHORA SIGNIFICATIVA"]
    if melhorias.empty:
        print("\nNenhuma melhoria estatisticamente significativa encontrada.")
    else:
        print("\n✅ MELHORIAS ESTATISTICAMENTE SIGNIFICATIVAS (p < 0.05):\n")
        destaque = melhorias.copy()
        destaque["IC95% Δ"] = destaque.apply(
            lambda row: f"[{row['IC95 low']:.4f}, {row['IC95 high']:.4f}]", axis=1
        )
        for coluna in ["Média Base", "Média Caso", "Δ Médio", "p-valor"]:
            destaque[coluna] = destaque[coluna].round(4)
        print(destaque[[
            "Variante", "Comparação", "Métrica", "Média Base", "Média Caso",
            "Δ Médio", "IC95% Δ", "p-valor",
        ]].to_string(index=False))

    print(f"\n{'─' * 88}")
    print("TABELA COMPLETA POR VARIANTE:")
    for variant in df["Variante"].drop_duplicates():
        print(f"\n── Variante: {variant.upper()} ──\n")
        tabela = df[df["Variante"] == variant].copy()
        tabela["IC95% Δ"] = tabela.apply(
            lambda row: f"[{row['IC95 low']:.4f}, {row['IC95 high']:.4f}]", axis=1
        )
        for coluna in ["Média Base", "Média Caso", "Δ Médio", "p-valor"]:
            tabela[coluna] = tabela[coluna].round(4)
        print(tabela[[
            "Comparação", "Métrica", "Média Base", "Média Caso", "Δ Médio",
            "IC95% Δ", "p-valor", "Status",
        ]].to_string(index=False))

    return df


## 7. Configuração manual A — 12 qubits × 1 feature por qubit

Requer `CLASSIC_CACHE_12`. Todos os parâmetros PQFM estão declarados na própria célula.
Altere `RUN_LABEL`, os parâmetros ou `VARIANTS_TO_RUN` sem modificar qualquer função.


In [10]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "12q_1f"
N_QUBITS = 12
FEATURES_PER_QUBIT = 1
AXES = ('x', 'y', 'z')
ENCODING_MODE = "shared_feature"
USE_GPU_STATEVECTOR = False
STATEVECTOR_DEVICE = None
CLASSIC_CACHE = CLASSIC_CACHE_12
FIXED_PHYS_NODES_FILE = FIXED_PHYS_NODES_12Q_FILE

# Remova/comente variantes para executar apenas o subconjunto desejado.
VARIANTS_TO_RUN = {
    "diagonal": {"keep_diagonal_terms": True, "keep_cross_terms": False},
    "cross-only": {"keep_diagonal_terms": False, "keep_cross_terms": True},
    "full": {"keep_diagonal_terms": True, "keep_cross_terms": True},
}
# ────────────────────────────────────────────────────────────────────

ensure_ibm_token()
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []
checkpoint_dir = Path("outputs/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
pqfm_fit_counter = 0
pqfm_fit_total = (
    (INNER_SPLITS + 1) * len(CLASSIC_CACHE["folds"]) * len(VARIANTS_TO_RUN)
)
print(f"Total de ajustes PQFM previstos nesta execução: {pqfm_fit_total}")

for cached_fold in CLASSIC_CACHE["folds"]:
    outer_fold = cached_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] outer fold {outer_fold}/{len(CLASSIC_CACHE['folds'])}")

    variant_best_params = {}

    # INNER CV: a instanciação e o fit da PQFM estão explícitos nesta célula.
    for variant, variant_options in VARIANTS_TO_RUN.items():
        scenario_scores = {
            "quantum_only": {candidate_key(params): [] for params in candidates},
            "hibrida": {candidate_key(params): [] for params in candidates},
        }

        for inner_data in cached_fold["inner_folds"]:
            inner_fold = inner_data["inner_fold"]
            X_inner_train = inner_data["X_train"]
            X_inner_validation = inner_data["X_validation"]
            y_inner_train = inner_data["y_train"]
            y_inner_validation = inner_data["y_validation"]

            pqfm = XYZProjectiveQFM(
                name_file=f"{RUN_LABEL}_outer{outer_fold:02d}_inner{inner_fold:02d}_{variant}",
                seed=RANDOM_STATE,
                ideal=False,
                simulation=True,
                fakebackend=False,
                shots=SHOTS,
                ibm_qpu=QPU_TARGET,
                qiskit_channel=QISKIT_CHANNEL,
                q_enc=N_QUBITS,
                features_per_qubit=FEATURES_PER_QUBIT,
                axes=AXES,
                encoding_mode=ENCODING_MODE,
                keep_diagonal_terms=variant_options["keep_diagonal_terms"],
                keep_cross_terms=variant_options["keep_cross_terms"],
                measure_cross_observables=False,
                use_gpu_statevector=USE_GPU_STATEVECTOR,
                statevector_device=STATEVECTOR_DEVICE,
                use_fixed_blocks=False,
                use_fixed_phys_nodes=FIXED_PHYS_NODES_FILE.exists(),
                fixed_phys_nodes_file=str(FIXED_PHYS_NODES_FILE),
                use_edge_error=False,
                output_root=f"outputs/pqfm/{RUN_LABEL}",
            )
            # Evita uma nova conexão: _load_real_backend() reutiliza este objeto.
            pqfm.real_backend = SHARED_REAL_BACKEND
            pqfm.service = SHARED_IBM_SERVICE

            pqfm_fit_counter += 1
            layout_mode = "FIXO" if FIXED_PHYS_NODES_FILE.exists() else "SELEÇÃO INICIAL"
            print(
                f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] START | "
                f"config={RUN_LABEL} | variante={variant} | "
                f"outer={outer_fold}/{len(CLASSIC_CACHE['folds'])} | "
                f"inner={inner_fold}/{INNER_SPLITS} | layout={layout_mode}"
            )
            pqfm_started_at = time.perf_counter()
            X_inner_quantum_values = pqfm.fit_transform(X_inner_train)

            if not FIXED_PHYS_NODES_FILE.exists():
                save_json([int(node) for node in pqfm.phys_nodes], FIXED_PHYS_NODES_FILE)
                print(
                    f"  Layout fixo salvo em {FIXED_PHYS_NODES_FILE}: "
                    f"{[int(node) for node in pqfm.phys_nodes]}"
                )

            X_validation_quantum_values = pqfm.transform(X_inner_validation)
            print(
                f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] DONE  | "
                f"outer={outer_fold} | inner={inner_fold} | variante={variant} | "
                f"tempo={time.perf_counter() - pqfm_started_at:.1f}s"
            )
            quantum_columns = [
                f"pqfm_{variant}_{i}" for i in range(X_inner_quantum_values.shape[1])
            ]
            X_inner_quantum = pd.DataFrame(
                X_inner_quantum_values, columns=quantum_columns
            ).reset_index(drop=True)
            X_validation_quantum = pd.DataFrame(
                X_validation_quantum_values, columns=quantum_columns
            ).reset_index(drop=True)
            X_inner_hybrid = pd.concat(
                [X_inner_train.reset_index(drop=True), X_inner_quantum], axis=1
            )
            X_validation_hybrid = pd.concat(
                [X_inner_validation.reset_index(drop=True), X_validation_quantum], axis=1
            )

            for params in candidates:
                key = candidate_key(params)
                scenario_scores["quantum_only"][key].append(
                    fit_and_score_classifier(
                        X_inner_quantum,
                        y_inner_train,
                        X_validation_quantum,
                        y_inner_validation,
                        params,
                    )
                )
                scenario_scores["hibrida"][key].append(
                    fit_and_score_classifier(
                        X_inner_hybrid,
                        y_inner_train,
                        X_validation_hybrid,
                        y_inner_validation,
                        params,
                    )
                )

        variant_best_params[variant] = {}
        for scenario in ("quantum_only", "hibrida"):
            mean_scores = {
                key: float(np.mean(values))
                for key, values in scenario_scores[scenario].items()
            }
            best_key = max(mean_scores, key=mean_scores.get)
            variant_best_params[variant][scenario] = json.loads(best_key)
            for key, values in scenario_scores[scenario].items():
                tuning_rows.append({
                    "config": RUN_LABEL,
                    "outer_fold": outer_fold,
                    "variant": variant,
                    "scenario": scenario,
                    "params": key,
                    "mean_inner_average_precision": np.mean(values),
                    "std_inner_average_precision": np.std(values, ddof=1),
                })

    # OUTER REFIT: nova PQFM ajustada em todo o outer_train, também explícita nesta célula.
    X_outer_train = cached_fold["X_outer_train"]
    X_outer_test = cached_fold["X_outer_test"]
    y_outer_train = cached_fold["y_outer_train"]
    y_outer_test = cached_fold["y_outer_test"]

    for variant, variant_options in VARIANTS_TO_RUN.items():
        pqfm = XYZProjectiveQFM(
            name_file=f"{RUN_LABEL}_outer{outer_fold:02d}_refit_{variant}",
            seed=RANDOM_STATE,
            ideal=False,
            simulation=True,
            fakebackend=False,
            shots=SHOTS,
            ibm_qpu=QPU_TARGET,
            qiskit_channel=QISKIT_CHANNEL,
            q_enc=N_QUBITS,
            features_per_qubit=FEATURES_PER_QUBIT,
            axes=AXES,
            encoding_mode=ENCODING_MODE,
            keep_diagonal_terms=variant_options["keep_diagonal_terms"],
            keep_cross_terms=variant_options["keep_cross_terms"],
            measure_cross_observables=False,
            use_gpu_statevector=USE_GPU_STATEVECTOR,
            statevector_device=STATEVECTOR_DEVICE,
            use_fixed_blocks=False,
            use_fixed_phys_nodes=FIXED_PHYS_NODES_FILE.exists(),
            fixed_phys_nodes_file=str(FIXED_PHYS_NODES_FILE),
            use_edge_error=False,
            output_root=f"outputs/pqfm/{RUN_LABEL}",
        )
        # Mesmo backend em memória e mesmos phys_nodes; MI/blocos continuam sendo reajustados.
        pqfm.real_backend = SHARED_REAL_BACKEND
        pqfm.service = SHARED_IBM_SERVICE

        pqfm_fit_counter += 1
        print(
            f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] START | "
            f"config={RUN_LABEL} | variante={variant} | "
            f"outer={outer_fold}/{len(CLASSIC_CACHE['folds'])} | "
            f"OUTER REFIT | layout=FIXO"
        )
        pqfm_started_at = time.perf_counter()
        X_outer_quantum_values = pqfm.fit_transform(X_outer_train)

        if not FIXED_PHYS_NODES_FILE.exists():
            save_json([int(node) for node in pqfm.phys_nodes], FIXED_PHYS_NODES_FILE)
            print(
                f"  Layout fixo salvo em {FIXED_PHYS_NODES_FILE}: "
                f"{[int(node) for node in pqfm.phys_nodes]}"
            )

        X_test_quantum_values = pqfm.transform(X_outer_test)
        print(
            f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] DONE  | "
            f"outer={outer_fold} | OUTER REFIT | variante={variant} | "
            f"tempo={time.perf_counter() - pqfm_started_at:.1f}s"
        )
        quantum_columns = [
            f"pqfm_{variant}_{i}" for i in range(X_outer_quantum_values.shape[1])
        ]
        X_outer_quantum = pd.DataFrame(
            X_outer_quantum_values, columns=quantum_columns
        ).reset_index(drop=True)
        X_test_quantum = pd.DataFrame(
            X_test_quantum_values, columns=quantum_columns
        ).reset_index(drop=True)
        X_outer_hybrid = pd.concat(
            [X_outer_train.reset_index(drop=True), X_outer_quantum], axis=1
        )
        X_test_hybrid = pd.concat(
            [X_outer_test.reset_index(drop=True), X_test_quantum], axis=1
        )

        # O clássico vem do cache e é associado às variantes para manter a comparação pareada.
        fold_rows.append({
            "config": RUN_LABEL,
            "variant": variant,
            "scenario": "classica",
            "outer_fold": outer_fold,
            "n_train": len(cached_fold["train_idx"]),
            "n_test": len(cached_fold["test_idx"]),
            "test_start_position": int(cached_fold["test_idx"][0]),
            "test_end_position": int(cached_fold["test_idx"][-1]),
            "best_params": json.dumps(cached_fold["best_classic_params"], sort_keys=True),
            **cached_fold["classic_metrics"],
        })

        for scenario, X_train_rep, X_test_rep in (
            ("quantum_only", X_outer_quantum, X_test_quantum),
            ("hibrida", X_outer_hybrid, X_test_hybrid),
        ):
            best_params = variant_best_params[variant][scenario]
            classifier = GradientBoostingClassifier(
                random_state=RANDOM_STATE, **best_params
            )
            classifier.fit(X_train_rep, y_outer_train)
            prediction = classifier.predict(X_test_rep)
            probability = classifier.predict_proba(X_test_rep)[:, 1]
            fold_rows.append({
                "config": RUN_LABEL,
                "variant": variant,
                "scenario": scenario,
                "outer_fold": outer_fold,
                "n_train": len(cached_fold["train_idx"]),
                "n_test": len(cached_fold["test_idx"]),
                "test_start_position": int(cached_fold["test_idx"][0]),
                "test_end_position": int(cached_fold["test_idx"][-1]),
                "best_params": json.dumps(best_params, sort_keys=True),
                **compute_metrics(y_outer_test, prediction, probability),
            })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        checkpoint_dir / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl", "wb"
    ) as handle:
        pickle.dump(checkpoint, handle)

experiment_result = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
EXPERIMENTS[RUN_LABEL] = experiment_result
EXPERIMENT_A_RESULT = experiment_result
print(f"Experimento {RUN_LABEL} concluído. Execute a próxima célula para a análise estatística.")


Credencial disponível somente no ambiente da sessão. Backend: ibm_marrakesh
Total de ajustes PQFM previstos nesta execução: 120

[12q_1f] outer fold 1/10
[PQFM 001/120] START | config=12q_1f | variante=diagonal | outer=1/10 | inner=1/3 | layout=SELEÇÃO INICIAL
Physical subgraph: [0, 1, 2, 3, 4, 16, 22, 23, 24, 25, 37, 45]
Number of blocks: 1
  Layout fixo salvo em outputs/fixed_phys_nodes/ibm_marrakesh_12q.json: [0, 1, 2, 3, 4, 16, 22, 23, 24, 25, 37, 45]
[PQFM 001/120] DONE  | outer=1 | inner=1 | variante=diagonal | tempo=8.4s
[PQFM 002/120] START | config=12q_1f | variante=diagonal | outer=1/10 | inner=2/3 | layout=FIXO
Physical subgraph: [0, 1, 2, 3, 4, 16, 22, 23, 24, 25, 37, 45]
Number of blocks: 1
[PQFM 002/120] DONE  | outer=1 | inner=2 | variante=diagonal | tempo=11.5s
[PQFM 003/120] START | config=12q_1f | variante=diagonal | outer=1/10 | inner=3/3 | layout=FIXO
Physical subgraph: [0, 1, 2, 3, 4, 16, 22, 23, 24, 25, 37, 45]
Number of blocks: 1
[PQFM 003/120] DONE  | outer=1 | 

,config,variant,scenario,outer_fold,n_train,n_test,test_start_position,test_end_position,best_params,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1,Specificity
0,12q_1f,diagonal,classica,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.974085,0.997575,0.944444,0.975309,0.963415,0.969325,0.750000
1,12q_1f,diagonal,quantum_only,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.927591,0.992247,0.855556,0.985915,0.853659,0.915033,0.875000
2,12q_1f,diagonal,hibrida,1,100,90,100,189,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.951220,0.994998,0.844444,0.972222,0.853659,0.909091,0.750000
3,12q_1f,cross-only,classica,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.974085,0.997575,0.944444,0.975309,0.963415,0.969325,0.750000
4,12q_1f,cross-only,quantum_only,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.903201,0.986398,0.944444,0.987342,0.951220,0.968944,0.875000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,12q_1f,cross-only,quantum_only,10,910,90,910,999,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.804409,0.916398,0.733333,0.824561,0.770492,0.796610,0.655172
86,12q_1f,cross-only,hibrida,10,910,90,910,999,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.859243,0.933889,0.744444,0.806452,0.819672,0.813008,0.586207
87,12q_1f,full,classica,10,910,90,910,999,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.822781,0.917903,0.733333,0.824561,0.770492,0.796610,0.655172
88,12q_1f,full,quantum_only,10,910,90,910,999,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.820803,0.919373,0.755556,0.854545,0.770492,0.810345,0.724138


,config,variant,scenario,metric,n_folds,mean,std,ci95_low,ci95_high
0,12q_1f,diagonal,classica,ROC-AUC,10,0.817293,0.095939,0.748662,0.885923
1,12q_1f,diagonal,quantum_only,ROC-AUC,10,0.793042,0.105215,0.717776,0.868308
2,12q_1f,diagonal,hibrida,ROC-AUC,10,0.803684,0.110846,0.724389,0.882979
3,12q_1f,cross-only,classica,ROC-AUC,10,0.817293,0.095939,0.748662,0.885923
4,12q_1f,cross-only,quantum_only,ROC-AUC,10,0.771833,0.107740,0.694761,0.848906
...,...,...,...,...,...,...,...,...,...
58,12q_1f,cross-only,quantum_only,Specificity,10,0.664240,0.130605,0.570810,0.757669
59,12q_1f,cross-only,hibrida,Specificity,10,0.656788,0.131802,0.562503,0.751074
60,12q_1f,full,classica,Specificity,10,0.690129,0.086768,0.628058,0.752199
61,12q_1f,full,quantum_only,Specificity,10,0.684805,0.164774,0.566933,0.802677


,config,variant,metric,baseline,challenger,n_pairs,mean_delta,wilcoxon_statistic,p_value
0,12q_1f,diagonal,ROC-AUC,classica,quantum_only,10,-0.024251,11.0,0.105469
1,12q_1f,diagonal,ROC-AUC,classica,hibrida,10,-0.013609,16.0,0.275391
2,12q_1f,diagonal,ROC-AUC,quantum_only,hibrida,10,0.010642,17.0,0.322266
3,12q_1f,cross-only,ROC-AUC,classica,quantum_only,10,-0.045459,5.0,0.019531
4,12q_1f,cross-only,ROC-AUC,classica,hibrida,10,-0.004687,23.0,0.695312
...,...,...,...,...,...,...,...,...,...
58,12q_1f,cross-only,Specificity,classica,hibrida,10,-0.033340,13.0,0.300781
59,12q_1f,cross-only,Specificity,quantum_only,hibrida,10,-0.007452,14.0,0.640625
60,12q_1f,full,Specificity,classica,quantum_only,10,-0.005324,27.0,1.000000
61,12q_1f,full,Specificity,classica,hibrida,10,-0.009235,21.0,0.910156


In [ ]:
# Análise separada, no formato compacto do notebook original.
df_stats_current = analise_estatistica(
    EXPERIMENT_A_RESULT["fold_results"],
    titulo=f"{EXPERIMENT_A_RESULT['config'].upper()} — 10 OUTER FOLDS",
)
if "STATISTICAL_ANALYSES" not in globals():
    STATISTICAL_ANALYSES = {}
STATISTICAL_ANALYSES[EXPERIMENT_A_RESULT["config"]] = df_stats_current


## 8. Configuração manual B — 12 qubits × 3 features por qubit

Requer `CLASSIC_CACHE_36`. Todos os parâmetros PQFM estão declarados na própria célula.


In [10]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "12q_3f"
N_QUBITS = 12
FEATURES_PER_QUBIT = 3
AXES = ('x', 'y', 'z')
ENCODING_MODE = "multi_axis"
USE_GPU_STATEVECTOR = False
STATEVECTOR_DEVICE = None
CLASSIC_CACHE = CLASSIC_CACHE_36
FIXED_PHYS_NODES_FILE = FIXED_PHYS_NODES_12Q_FILE

# Remova/comente variantes para executar apenas o subconjunto desejado.
VARIANTS_TO_RUN = {
    "diagonal": {"keep_diagonal_terms": True, "keep_cross_terms": False},
    "cross-only": {"keep_diagonal_terms": False, "keep_cross_terms": True},
    "full": {"keep_diagonal_terms": True, "keep_cross_terms": True},
}
# ────────────────────────────────────────────────────────────────────

ensure_ibm_token()
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []
checkpoint_dir = Path("outputs/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
pqfm_fit_counter = 0
pqfm_fit_total = (
    (INNER_SPLITS + 1) * len(CLASSIC_CACHE["folds"]) * len(VARIANTS_TO_RUN)
)
print(f"Total de ajustes PQFM previstos nesta execução: {pqfm_fit_total}")

for cached_fold in CLASSIC_CACHE["folds"]:
    outer_fold = cached_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] outer fold {outer_fold}/{len(CLASSIC_CACHE['folds'])}")

    variant_best_params = {}

    # INNER CV: a instanciação e o fit da PQFM estão explícitos nesta célula.
    for variant, variant_options in VARIANTS_TO_RUN.items():
        scenario_scores = {
            "quantum_only": {candidate_key(params): [] for params in candidates},
            "hibrida": {candidate_key(params): [] for params in candidates},
        }

        for inner_data in cached_fold["inner_folds"]:
            inner_fold = inner_data["inner_fold"]
            X_inner_train = inner_data["X_train"]
            X_inner_validation = inner_data["X_validation"]
            y_inner_train = inner_data["y_train"]
            y_inner_validation = inner_data["y_validation"]

            pqfm = XYZProjectiveQFM(
                name_file=f"{RUN_LABEL}_outer{outer_fold:02d}_inner{inner_fold:02d}_{variant}",
                seed=RANDOM_STATE,
                ideal=False,
                simulation=True,
                fakebackend=False,
                shots=SHOTS,
                ibm_qpu=QPU_TARGET,
                qiskit_channel=QISKIT_CHANNEL,
                q_enc=N_QUBITS,
                features_per_qubit=FEATURES_PER_QUBIT,
                axes=AXES,
                encoding_mode=ENCODING_MODE,
                keep_diagonal_terms=variant_options["keep_diagonal_terms"],
                keep_cross_terms=variant_options["keep_cross_terms"],
                measure_cross_observables=False,
                use_gpu_statevector=USE_GPU_STATEVECTOR,
                statevector_device=STATEVECTOR_DEVICE,
                use_fixed_blocks=False,
                use_fixed_phys_nodes=FIXED_PHYS_NODES_FILE.exists(),
                fixed_phys_nodes_file=str(FIXED_PHYS_NODES_FILE),
                use_edge_error=False,
                output_root=f"outputs/pqfm/{RUN_LABEL}",
            )
            # Evita uma nova conexão: _load_real_backend() reutiliza este objeto.
            pqfm.real_backend = SHARED_REAL_BACKEND
            pqfm.service = SHARED_IBM_SERVICE

            pqfm_fit_counter += 1
            layout_mode = "FIXO" if FIXED_PHYS_NODES_FILE.exists() else "SELEÇÃO INICIAL"
            print(
                f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] START | "
                f"config={RUN_LABEL} | variante={variant} | "
                f"outer={outer_fold}/{len(CLASSIC_CACHE['folds'])} | "
                f"inner={inner_fold}/{INNER_SPLITS} | layout={layout_mode}"
            )
            pqfm_started_at = time.perf_counter()
            X_inner_quantum_values = pqfm.fit_transform(X_inner_train)

            if not FIXED_PHYS_NODES_FILE.exists():
                save_json([int(node) for node in pqfm.phys_nodes], FIXED_PHYS_NODES_FILE)
                print(
                    f"  Layout fixo salvo em {FIXED_PHYS_NODES_FILE}: "
                    f"{[int(node) for node in pqfm.phys_nodes]}"
                )

            X_validation_quantum_values = pqfm.transform(X_inner_validation)
            print(
                f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] DONE  | "
                f"outer={outer_fold} | inner={inner_fold} | variante={variant} | "
                f"tempo={time.perf_counter() - pqfm_started_at:.1f}s"
            )
            quantum_columns = [
                f"pqfm_{variant}_{i}" for i in range(X_inner_quantum_values.shape[1])
            ]
            X_inner_quantum = pd.DataFrame(
                X_inner_quantum_values, columns=quantum_columns
            ).reset_index(drop=True)
            X_validation_quantum = pd.DataFrame(
                X_validation_quantum_values, columns=quantum_columns
            ).reset_index(drop=True)
            X_inner_hybrid = pd.concat(
                [X_inner_train.reset_index(drop=True), X_inner_quantum], axis=1
            )
            X_validation_hybrid = pd.concat(
                [X_inner_validation.reset_index(drop=True), X_validation_quantum], axis=1
            )

            for params in candidates:
                key = candidate_key(params)
                scenario_scores["quantum_only"][key].append(
                    fit_and_score_classifier(
                        X_inner_quantum,
                        y_inner_train,
                        X_validation_quantum,
                        y_inner_validation,
                        params,
                    )
                )
                scenario_scores["hibrida"][key].append(
                    fit_and_score_classifier(
                        X_inner_hybrid,
                        y_inner_train,
                        X_validation_hybrid,
                        y_inner_validation,
                        params,
                    )
                )

        variant_best_params[variant] = {}
        for scenario in ("quantum_only", "hibrida"):
            mean_scores = {
                key: float(np.mean(values))
                for key, values in scenario_scores[scenario].items()
            }
            best_key = max(mean_scores, key=mean_scores.get)
            variant_best_params[variant][scenario] = json.loads(best_key)
            for key, values in scenario_scores[scenario].items():
                tuning_rows.append({
                    "config": RUN_LABEL,
                    "outer_fold": outer_fold,
                    "variant": variant,
                    "scenario": scenario,
                    "params": key,
                    "mean_inner_average_precision": np.mean(values),
                    "std_inner_average_precision": np.std(values, ddof=1),
                })

    # OUTER REFIT: nova PQFM ajustada em todo o outer_train, também explícita nesta célula.
    X_outer_train = cached_fold["X_outer_train"]
    X_outer_test = cached_fold["X_outer_test"]
    y_outer_train = cached_fold["y_outer_train"]
    y_outer_test = cached_fold["y_outer_test"]

    for variant, variant_options in VARIANTS_TO_RUN.items():
        pqfm = XYZProjectiveQFM(
            name_file=f"{RUN_LABEL}_outer{outer_fold:02d}_refit_{variant}",
            seed=RANDOM_STATE,
            ideal=False,
            simulation=True,
            fakebackend=False,
            shots=SHOTS,
            ibm_qpu=QPU_TARGET,
            qiskit_channel=QISKIT_CHANNEL,
            q_enc=N_QUBITS,
            features_per_qubit=FEATURES_PER_QUBIT,
            axes=AXES,
            encoding_mode=ENCODING_MODE,
            keep_diagonal_terms=variant_options["keep_diagonal_terms"],
            keep_cross_terms=variant_options["keep_cross_terms"],
            measure_cross_observables=False,
            use_gpu_statevector=USE_GPU_STATEVECTOR,
            statevector_device=STATEVECTOR_DEVICE,
            use_fixed_blocks=False,
            use_fixed_phys_nodes=FIXED_PHYS_NODES_FILE.exists(),
            fixed_phys_nodes_file=str(FIXED_PHYS_NODES_FILE),
            use_edge_error=False,
            output_root=f"outputs/pqfm/{RUN_LABEL}",
        )
        # Mesmo backend em memória e mesmos phys_nodes; MI/blocos continuam sendo reajustados.
        pqfm.real_backend = SHARED_REAL_BACKEND
        pqfm.service = SHARED_IBM_SERVICE

        pqfm_fit_counter += 1
        print(
            f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] START | "
            f"config={RUN_LABEL} | variante={variant} | "
            f"outer={outer_fold}/{len(CLASSIC_CACHE['folds'])} | "
            f"OUTER REFIT | layout=FIXO"
        )
        pqfm_started_at = time.perf_counter()
        X_outer_quantum_values = pqfm.fit_transform(X_outer_train)

        if not FIXED_PHYS_NODES_FILE.exists():
            save_json([int(node) for node in pqfm.phys_nodes], FIXED_PHYS_NODES_FILE)
            print(
                f"  Layout fixo salvo em {FIXED_PHYS_NODES_FILE}: "
                f"{[int(node) for node in pqfm.phys_nodes]}"
            )

        X_test_quantum_values = pqfm.transform(X_outer_test)
        print(
            f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] DONE  | "
            f"outer={outer_fold} | OUTER REFIT | variante={variant} | "
            f"tempo={time.perf_counter() - pqfm_started_at:.1f}s"
        )
        quantum_columns = [
            f"pqfm_{variant}_{i}" for i in range(X_outer_quantum_values.shape[1])
        ]
        X_outer_quantum = pd.DataFrame(
            X_outer_quantum_values, columns=quantum_columns
        ).reset_index(drop=True)
        X_test_quantum = pd.DataFrame(
            X_test_quantum_values, columns=quantum_columns
        ).reset_index(drop=True)
        X_outer_hybrid = pd.concat(
            [X_outer_train.reset_index(drop=True), X_outer_quantum], axis=1
        )
        X_test_hybrid = pd.concat(
            [X_outer_test.reset_index(drop=True), X_test_quantum], axis=1
        )

        # O clássico vem do cache e é associado às variantes para manter a comparação pareada.
        fold_rows.append({
            "config": RUN_LABEL,
            "variant": variant,
            "scenario": "classica",
            "outer_fold": outer_fold,
            "n_train": len(cached_fold["train_idx"]),
            "n_test": len(cached_fold["test_idx"]),
            "test_start_position": int(cached_fold["test_idx"][0]),
            "test_end_position": int(cached_fold["test_idx"][-1]),
            "best_params": json.dumps(cached_fold["best_classic_params"], sort_keys=True),
            **cached_fold["classic_metrics"],
        })

        for scenario, X_train_rep, X_test_rep in (
            ("quantum_only", X_outer_quantum, X_test_quantum),
            ("hibrida", X_outer_hybrid, X_test_hybrid),
        ):
            best_params = variant_best_params[variant][scenario]
            classifier = GradientBoostingClassifier(
                random_state=RANDOM_STATE, **best_params
            )
            classifier.fit(X_train_rep, y_outer_train)
            prediction = classifier.predict(X_test_rep)
            probability = classifier.predict_proba(X_test_rep)[:, 1]
            fold_rows.append({
                "config": RUN_LABEL,
                "variant": variant,
                "scenario": scenario,
                "outer_fold": outer_fold,
                "n_train": len(cached_fold["train_idx"]),
                "n_test": len(cached_fold["test_idx"]),
                "test_start_position": int(cached_fold["test_idx"][0]),
                "test_end_position": int(cached_fold["test_idx"][-1]),
                "best_params": json.dumps(best_params, sort_keys=True),
                **compute_metrics(y_outer_test, prediction, probability),
            })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        checkpoint_dir / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl", "wb"
    ) as handle:
        pickle.dump(checkpoint, handle)

experiment_result = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
EXPERIMENTS[RUN_LABEL] = experiment_result
EXPERIMENT_B_RESULT = experiment_result
print(f"Experimento {RUN_LABEL} concluído. Execute a próxima célula para a análise estatística.")


Credencial disponível somente no ambiente da sessão. Backend: ibm_marrakesh
Total de ajustes PQFM previstos nesta execução: 120

[12q_3f] outer fold 1/10
[PQFM 001/120] START | config=12q_3f | variante=diagonal | outer=1/10 | inner=1/3 | layout=FIXO
Physical subgraph: [0, 1, 2, 3, 4, 16, 22, 23, 24, 25, 37, 45]
Number of blocks: 1
[PQFM 001/120] DONE  | outer=1 | inner=1 | variante=diagonal | tempo=8.4s
[PQFM 002/120] START | config=12q_3f | variante=diagonal | outer=1/10 | inner=2/3 | layout=FIXO
Physical subgraph: [0, 1, 2, 3, 4, 16, 22, 23, 24, 25, 37, 45]
Number of blocks: 1
[PQFM 002/120] DONE  | outer=1 | inner=2 | variante=diagonal | tempo=14.0s
[PQFM 003/120] START | config=12q_3f | variante=diagonal | outer=1/10 | inner=3/3 | layout=FIXO
Physical subgraph: [0, 1, 2, 3, 4, 16, 22, 23, 24, 25, 37, 45]
Number of blocks: 1
[PQFM 003/120] DONE  | outer=1 | inner=3 | variante=diagonal | tempo=15.9s
[PQFM 004/120] START | config=12q_3f | variante=cross-only | outer=1/10 | inner=1/3 |

In [11]:
# Análise separada, no formato compacto do notebook original.
df_stats_current = analise_estatistica(
    EXPERIMENT_B_RESULT["fold_results"],
    titulo=f"{EXPERIMENT_B_RESULT['config'].upper()} — 10 OUTER FOLDS",
)
if "STATISTICAL_ANALYSES" not in globals():
    STATISTICAL_ANALYSES = {}
STATISTICAL_ANALYSES[EXPERIMENT_B_RESULT["config"]] = df_stats_current



  12Q_3F — 10 OUTER FOLDS

Nenhuma melhoria estatisticamente significativa encontrada.

────────────────────────────────────────────────────────────────────────────────────────
TABELA COMPLETA POR VARIANTE:

── Variante: DIAGONAL ──

              Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor                 Status
Clássico vs Quantum-Only     ROC-AUC      0.8109      0.7756  -0.0353  [-0.0914, 0.0117]   0.3750     ↓ Piora (não sig.)
     Clássico vs Híbrido     ROC-AUC      0.8109      0.8112   0.0003  [-0.0483, 0.0445]   0.7695   ↑ Melhora (não sig.)
Clássico vs Quantum-Only      PR-AUC      0.8854      0.8858   0.0004  [-0.0270, 0.0306]   0.8457   ↑ Melhora (não sig.)
     Clássico vs Híbrido      PR-AUC      0.8854      0.8972   0.0118  [-0.0069, 0.0323]   0.5566   ↑ Melhora (não sig.)
Clássico vs Quantum-Only    Accuracy      0.7511      0.6778  -0.0733  [-0.1745, 0.0011]   0.1934     ↓ Piora (não sig.)
     Clássico vs Híbrido    Accuracy   

## 9. Configuração manual C — 18 qubits × 2 features por qubit, GPU

Requer `CLASSIC_CACHE_36`. Todos os parâmetros PQFM e GPU estão declarados na própria célula.


In [10]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "18q_2f_gpu"
N_QUBITS = 18
FEATURES_PER_QUBIT = 2
AXES = ('x', 'y')
ENCODING_MODE = "multi_axis"
USE_GPU_STATEVECTOR = True
STATEVECTOR_DEVICE = 'GPU'
CLASSIC_CACHE = CLASSIC_CACHE_36
FIXED_PHYS_NODES_FILE = FIXED_PHYS_NODES_18Q_FILE

# Remova/comente variantes para executar apenas o subconjunto desejado.
VARIANTS_TO_RUN = {
    "diagonal": {"keep_diagonal_terms": True, "keep_cross_terms": False},
    "cross-only": {"keep_diagonal_terms": False, "keep_cross_terms": True},
    "full": {"keep_diagonal_terms": True, "keep_cross_terms": True},
}
# ────────────────────────────────────────────────────────────────────

ensure_ibm_token()
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []
checkpoint_dir = Path("outputs/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
pqfm_fit_counter = 0
pqfm_fit_total = (
    (INNER_SPLITS + 1) * len(CLASSIC_CACHE["folds"]) * len(VARIANTS_TO_RUN)
)
print(f"Total de ajustes PQFM previstos nesta execução: {pqfm_fit_total}")

for cached_fold in CLASSIC_CACHE["folds"]:
    outer_fold = cached_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] outer fold {outer_fold}/{len(CLASSIC_CACHE['folds'])}")

    variant_best_params = {}

    # INNER CV: a instanciação e o fit da PQFM estão explícitos nesta célula.
    for variant, variant_options in VARIANTS_TO_RUN.items():
        scenario_scores = {
            "quantum_only": {candidate_key(params): [] for params in candidates},
            "hibrida": {candidate_key(params): [] for params in candidates},
        }

        for inner_data in cached_fold["inner_folds"]:
            inner_fold = inner_data["inner_fold"]
            X_inner_train = inner_data["X_train"]
            X_inner_validation = inner_data["X_validation"]
            y_inner_train = inner_data["y_train"]
            y_inner_validation = inner_data["y_validation"]

            pqfm = XYZProjectiveQFM(
                name_file=f"{RUN_LABEL}_outer{outer_fold:02d}_inner{inner_fold:02d}_{variant}",
                seed=RANDOM_STATE,
                ideal=False,
                simulation=True,
                fakebackend=False,
                shots=SHOTS,
                ibm_qpu=QPU_TARGET,
                qiskit_channel=QISKIT_CHANNEL,
                q_enc=N_QUBITS,
                features_per_qubit=FEATURES_PER_QUBIT,
                axes=AXES,
                encoding_mode=ENCODING_MODE,
                keep_diagonal_terms=variant_options["keep_diagonal_terms"],
                keep_cross_terms=variant_options["keep_cross_terms"],
                measure_cross_observables=False,
                use_gpu_statevector=USE_GPU_STATEVECTOR,
                statevector_device=STATEVECTOR_DEVICE,
                use_fixed_blocks=False,
                use_fixed_phys_nodes=FIXED_PHYS_NODES_FILE.exists(),
                fixed_phys_nodes_file=str(FIXED_PHYS_NODES_FILE),
                use_edge_error=False,
                output_root=f"outputs/pqfm/{RUN_LABEL}",
            )
            # Evita uma nova conexão: _load_real_backend() reutiliza este objeto.
            pqfm.real_backend = SHARED_REAL_BACKEND
            pqfm.service = SHARED_IBM_SERVICE

            pqfm_fit_counter += 1
            layout_mode = "FIXO" if FIXED_PHYS_NODES_FILE.exists() else "SELEÇÃO INICIAL"
            print(
                f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] START | "
                f"config={RUN_LABEL} | variante={variant} | "
                f"outer={outer_fold}/{len(CLASSIC_CACHE['folds'])} | "
                f"inner={inner_fold}/{INNER_SPLITS} | layout={layout_mode}"
            )
            pqfm_started_at = time.perf_counter()
            X_inner_quantum_values = pqfm.fit_transform(X_inner_train)

            if not FIXED_PHYS_NODES_FILE.exists():
                save_json([int(node) for node in pqfm.phys_nodes], FIXED_PHYS_NODES_FILE)
                print(
                    f"  Layout fixo salvo em {FIXED_PHYS_NODES_FILE}: "
                    f"{[int(node) for node in pqfm.phys_nodes]}"
                )

            X_validation_quantum_values = pqfm.transform(X_inner_validation)
            print(
                f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] DONE  | "
                f"outer={outer_fold} | inner={inner_fold} | variante={variant} | "
                f"tempo={time.perf_counter() - pqfm_started_at:.1f}s"
            )
            quantum_columns = [
                f"pqfm_{variant}_{i}" for i in range(X_inner_quantum_values.shape[1])
            ]
            X_inner_quantum = pd.DataFrame(
                X_inner_quantum_values, columns=quantum_columns
            ).reset_index(drop=True)
            X_validation_quantum = pd.DataFrame(
                X_validation_quantum_values, columns=quantum_columns
            ).reset_index(drop=True)
            X_inner_hybrid = pd.concat(
                [X_inner_train.reset_index(drop=True), X_inner_quantum], axis=1
            )
            X_validation_hybrid = pd.concat(
                [X_inner_validation.reset_index(drop=True), X_validation_quantum], axis=1
            )

            for params in candidates:
                key = candidate_key(params)
                scenario_scores["quantum_only"][key].append(
                    fit_and_score_classifier(
                        X_inner_quantum,
                        y_inner_train,
                        X_validation_quantum,
                        y_inner_validation,
                        params,
                    )
                )
                scenario_scores["hibrida"][key].append(
                    fit_and_score_classifier(
                        X_inner_hybrid,
                        y_inner_train,
                        X_validation_hybrid,
                        y_inner_validation,
                        params,
                    )
                )

        variant_best_params[variant] = {}
        for scenario in ("quantum_only", "hibrida"):
            mean_scores = {
                key: float(np.mean(values))
                for key, values in scenario_scores[scenario].items()
            }
            best_key = max(mean_scores, key=mean_scores.get)
            variant_best_params[variant][scenario] = json.loads(best_key)
            for key, values in scenario_scores[scenario].items():
                tuning_rows.append({
                    "config": RUN_LABEL,
                    "outer_fold": outer_fold,
                    "variant": variant,
                    "scenario": scenario,
                    "params": key,
                    "mean_inner_average_precision": np.mean(values),
                    "std_inner_average_precision": np.std(values, ddof=1),
                })

    # OUTER REFIT: nova PQFM ajustada em todo o outer_train, também explícita nesta célula.
    X_outer_train = cached_fold["X_outer_train"]
    X_outer_test = cached_fold["X_outer_test"]
    y_outer_train = cached_fold["y_outer_train"]
    y_outer_test = cached_fold["y_outer_test"]

    for variant, variant_options in VARIANTS_TO_RUN.items():
        pqfm = XYZProjectiveQFM(
            name_file=f"{RUN_LABEL}_outer{outer_fold:02d}_refit_{variant}",
            seed=RANDOM_STATE,
            ideal=False,
            simulation=True,
            fakebackend=False,
            shots=SHOTS,
            ibm_qpu=QPU_TARGET,
            qiskit_channel=QISKIT_CHANNEL,
            q_enc=N_QUBITS,
            features_per_qubit=FEATURES_PER_QUBIT,
            axes=AXES,
            encoding_mode=ENCODING_MODE,
            keep_diagonal_terms=variant_options["keep_diagonal_terms"],
            keep_cross_terms=variant_options["keep_cross_terms"],
            measure_cross_observables=False,
            use_gpu_statevector=USE_GPU_STATEVECTOR,
            statevector_device=STATEVECTOR_DEVICE,
            use_fixed_blocks=False,
            use_fixed_phys_nodes=FIXED_PHYS_NODES_FILE.exists(),
            fixed_phys_nodes_file=str(FIXED_PHYS_NODES_FILE),
            use_edge_error=False,
            output_root=f"outputs/pqfm/{RUN_LABEL}",
        )
        # Mesmo backend em memória e mesmos phys_nodes; MI/blocos continuam sendo reajustados.
        pqfm.real_backend = SHARED_REAL_BACKEND
        pqfm.service = SHARED_IBM_SERVICE

        pqfm_fit_counter += 1
        print(
            f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] START | "
            f"config={RUN_LABEL} | variante={variant} | "
            f"outer={outer_fold}/{len(CLASSIC_CACHE['folds'])} | "
            f"OUTER REFIT | layout=FIXO"
        )
        pqfm_started_at = time.perf_counter()
        X_outer_quantum_values = pqfm.fit_transform(X_outer_train)

        if not FIXED_PHYS_NODES_FILE.exists():
            save_json([int(node) for node in pqfm.phys_nodes], FIXED_PHYS_NODES_FILE)
            print(
                f"  Layout fixo salvo em {FIXED_PHYS_NODES_FILE}: "
                f"{[int(node) for node in pqfm.phys_nodes]}"
            )

        X_test_quantum_values = pqfm.transform(X_outer_test)
        print(
            f"[PQFM {pqfm_fit_counter:03d}/{pqfm_fit_total:03d}] DONE  | "
            f"outer={outer_fold} | OUTER REFIT | variante={variant} | "
            f"tempo={time.perf_counter() - pqfm_started_at:.1f}s"
        )
        quantum_columns = [
            f"pqfm_{variant}_{i}" for i in range(X_outer_quantum_values.shape[1])
        ]
        X_outer_quantum = pd.DataFrame(
            X_outer_quantum_values, columns=quantum_columns
        ).reset_index(drop=True)
        X_test_quantum = pd.DataFrame(
            X_test_quantum_values, columns=quantum_columns
        ).reset_index(drop=True)
        X_outer_hybrid = pd.concat(
            [X_outer_train.reset_index(drop=True), X_outer_quantum], axis=1
        )
        X_test_hybrid = pd.concat(
            [X_outer_test.reset_index(drop=True), X_test_quantum], axis=1
        )

        # O clássico vem do cache e é associado às variantes para manter a comparação pareada.
        fold_rows.append({
            "config": RUN_LABEL,
            "variant": variant,
            "scenario": "classica",
            "outer_fold": outer_fold,
            "n_train": len(cached_fold["train_idx"]),
            "n_test": len(cached_fold["test_idx"]),
            "test_start_position": int(cached_fold["test_idx"][0]),
            "test_end_position": int(cached_fold["test_idx"][-1]),
            "best_params": json.dumps(cached_fold["best_classic_params"], sort_keys=True),
            **cached_fold["classic_metrics"],
        })

        for scenario, X_train_rep, X_test_rep in (
            ("quantum_only", X_outer_quantum, X_test_quantum),
            ("hibrida", X_outer_hybrid, X_test_hybrid),
        ):
            best_params = variant_best_params[variant][scenario]
            classifier = GradientBoostingClassifier(
                random_state=RANDOM_STATE, **best_params
            )
            classifier.fit(X_train_rep, y_outer_train)
            prediction = classifier.predict(X_test_rep)
            probability = classifier.predict_proba(X_test_rep)[:, 1]
            fold_rows.append({
                "config": RUN_LABEL,
                "variant": variant,
                "scenario": scenario,
                "outer_fold": outer_fold,
                "n_train": len(cached_fold["train_idx"]),
                "n_test": len(cached_fold["test_idx"]),
                "test_start_position": int(cached_fold["test_idx"][0]),
                "test_end_position": int(cached_fold["test_idx"][-1]),
                "best_params": json.dumps(best_params, sort_keys=True),
                **compute_metrics(y_outer_test, prediction, probability),
            })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        checkpoint_dir / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl", "wb"
    ) as handle:
        pickle.dump(checkpoint, handle)

experiment_result = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
EXPERIMENTS[RUN_LABEL] = experiment_result
EXPERIMENT_C_RESULT = experiment_result
print(f"Experimento {RUN_LABEL} concluído. Execute a próxima célula para a análise estatística.")


Credencial disponível somente no ambiente da sessão. Backend: ibm_marrakesh
Total de ajustes PQFM previstos nesta execução: 120

[18q_2f_gpu] outer fold 1/10
[PQFM 001/120] START | config=18q_2f_gpu | variante=diagonal | outer=1/10 | inner=1/3 | layout=SELEÇÃO INICIAL
Physical subgraph: [87, 88, 89, 91, 92, 97, 98, 104, 105, 106, 107, 108, 109, 110, 111, 112, 118, 129]
Number of blocks: 1
  Layout fixo salvo em outputs/fixed_phys_nodes/ibm_marrakesh_18q.json: [87, 88, 89, 91, 92, 97, 98, 104, 105, 106, 107, 108, 109, 110, 111, 112, 118, 129]
[PQFM 001/120] DONE  | outer=1 | inner=1 | variante=diagonal | tempo=15.4s
[PQFM 002/120] START | config=18q_2f_gpu | variante=diagonal | outer=1/10 | inner=2/3 | layout=FIXO
Physical subgraph: [87, 88, 89, 91, 92, 97, 98, 104, 105, 106, 107, 108, 109, 110, 111, 112, 118, 129]
Number of blocks: 1
[PQFM 002/120] DONE  | outer=1 | inner=2 | variante=diagonal | tempo=16.4s
[PQFM 003/120] START | config=18q_2f_gpu | variante=diagonal | outer=1/10 | inn

In [11]:
# Análise separada, no formato compacto do notebook original.
df_stats_current = analise_estatistica(
    EXPERIMENT_C_RESULT["fold_results"],
    titulo=f"{EXPERIMENT_C_RESULT['config'].upper()} — 10 OUTER FOLDS",
)
if "STATISTICAL_ANALYSES" not in globals():
    STATISTICAL_ANALYSES = {}
STATISTICAL_ANALYSES[EXPERIMENT_C_RESULT["config"]] = df_stats_current



  18Q_2F_GPU — 10 OUTER FOLDS

Nenhuma melhoria estatisticamente significativa encontrada.

────────────────────────────────────────────────────────────────────────────────────────
TABELA COMPLETA POR VARIANTE:

── Variante: DIAGONAL ──

              Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor                 Status
Clássico vs Quantum-Only     ROC-AUC      0.8109      0.7838  -0.0271  [-0.0615, 0.0077]   0.1934     ↓ Piora (não sig.)
     Clássico vs Híbrido     ROC-AUC      0.8109      0.8153   0.0044  [-0.0142, 0.0245]   0.8457   ↑ Melhora (não sig.)
Clássico vs Quantum-Only      PR-AUC      0.8854      0.8722  -0.0132  [-0.0369, 0.0125]   0.1602     ↓ Piora (não sig.)
     Clássico vs Híbrido      PR-AUC      0.8854      0.8890   0.0036  [-0.0067, 0.0157]   1.0000   ↑ Melhora (não sig.)
Clássico vs Quantum-Only    Accuracy      0.7511      0.7100  -0.0411 [-0.0667, -0.0167]   0.0195 ⚠️ PIORA SIGNIFICATIVA
     Clássico vs Híbrido    Accurac

## 10. Heisenberg — 19 qubits × 36 features (2 blocos), GPU

A cadeia de 19 qubits possui 18 arestas por bloco. As 36 features selecionadas por SHAP ocupam exatamente dois blocos, sem padding.

A Heisenberg não usa MI nem aprende parâmetros a partir dos valores de treino. Por isso, o circuito é ajustado uma única vez, fora da validação. As transformações quânticas são pré-calculadas e armazenadas em cache para cada representação leakage-safe produzida por preprocessing + SHAP nos respectivos folds. A célula de tuning seguinte não executa nem reajusta a PQFM.


In [10]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "19q_2f_heisenberg_gpu"
N_QUBITS = 19
N_SHAP_FEATURES = 36
HEISENBERG_R = 2
HEISENBERG_ALPHA = 0.1
HEISENBERG_SHOTS = 4096
USE_TANH_SCALING = True
MEASURE_2LOCAL_DIAGONAL = False
CLASSIC_CACHE = CLASSIC_CACHE_36
# ────────────────────────────────────────────────────────────────────

HEISENBERG_CACHE_KEY = (
    RUN_LABEL, id(CLASSIC_CACHE), RANDOM_STATE,
    N_QUBITS, N_SHAP_FEATURES, HEISENBERG_R,
    HEISENBERG_ALPHA, HEISENBERG_SHOTS, USE_TANH_SCALING,
    MEASURE_2LOCAL_DIAGONAL,
)

cache_is_current = (
    "HEISENBERG_CACHE_19Q_36F" in globals()
    and HEISENBERG_CACHE_19Q_36F.get("cache_key") == HEISENBERG_CACHE_KEY
)

if cache_is_current:
    print("Cache Heisenberg 19q × 36f já existe; nenhuma simulação quântica foi repetida.")
else:
    # O fit recebe somente uma matriz dummy para congelar a dimensionalidade e o circuito.
    # Nenhuma estatística ou valor do dataset participa deste ajuste.
    HEISENBERG_PQFM_19Q = HeisenbergProjectiveQFM(
        name_file=RUN_LABEL,
        seed=RANDOM_STATE,
        ideal=True,
        simulation=True,
        fakebackend=False,
        shots=HEISENBERG_SHOTS,
        q_enc=N_QUBITS,
        R=HEISENBERG_R,
        alpha=HEISENBERG_ALPHA,
        use_tanh_scaling=USE_TANH_SCALING,
        measure_2local_diagonal=MEASURE_2LOCAL_DIAGONAL,
        use_gpu_statevector=True,
        statevector_device="GPU",
        output_root=f"outputs/pqfm/{RUN_LABEL}",
    )

    print("Preparando uma única vez o circuito Heisenberg na GPU...")
    HEISENBERG_PQFM_19Q.fit(np.zeros((1, N_SHAP_FEATURES), dtype=float))
    assert HEISENBERG_PQFM_19Q.theta_info["features_per_block"] == 18
    assert HEISENBERG_PQFM_19Q.theta_info["num_blocks"] == 2
    assert HEISENBERG_PQFM_19Q.theta_info["total_slots"] == 36
    print("Circuito congelado: 19 qubits | 18 arestas/bloco | 2 blocos | GPU.")

    heisenberg_cached_folds = []
    transform_counter = 0
    transform_total = len(CLASSIC_CACHE["folds"]) * (INNER_SPLITS + 1)

    for cached_fold in CLASSIC_CACHE["folds"]:
        outer_fold = cached_fold["outer_fold"]
        inner_quantum = []

        for inner_data in cached_fold["inner_folds"]:
            inner_fold = inner_data["inner_fold"]
            n_inner_train = len(inner_data["X_train"])
            X_inner_all = pd.concat(
                [inner_data["X_train"], inner_data["X_validation"]],
                ignore_index=True,
            )

            transform_counter += 1
            print(
                f"[Heisenberg {transform_counter:02d}/{transform_total:02d}] "
                f"outer={outer_fold}/10 | inner={inner_fold}/3 | "
                f"amostras={len(X_inner_all)} | GPU"
            )
            started_at = time.perf_counter()
            Xq_inner_all_values = HEISENBERG_PQFM_19Q.transform(X_inner_all)
            quantum_columns = [
                f"heisenberg_q_{i}" for i in range(Xq_inner_all_values.shape[1])
            ]
            Xq_inner_all = pd.DataFrame(Xq_inner_all_values, columns=quantum_columns)
            inner_quantum.append({
                "inner_fold": inner_fold,
                "X_train_quantum": Xq_inner_all.iloc[:n_inner_train].reset_index(drop=True),
                "X_validation_quantum": Xq_inner_all.iloc[n_inner_train:].reset_index(drop=True),
            })
            print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        n_outer_train = len(cached_fold["X_outer_train"])
        X_outer_all = pd.concat(
            [cached_fold["X_outer_train"], cached_fold["X_outer_test"]],
            ignore_index=True,
        )
        transform_counter += 1
        print(
            f"[Heisenberg {transform_counter:02d}/{transform_total:02d}] "
            f"outer={outer_fold}/10 | OUTER | amostras={len(X_outer_all)} | GPU"
        )
        started_at = time.perf_counter()
        Xq_outer_all_values = HEISENBERG_PQFM_19Q.transform(X_outer_all)
        quantum_columns = [
            f"heisenberg_q_{i}" for i in range(Xq_outer_all_values.shape[1])
        ]
        Xq_outer_all = pd.DataFrame(Xq_outer_all_values, columns=quantum_columns)
        print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        heisenberg_cached_folds.append({
            "classic_fold": cached_fold,
            "inner_quantum": inner_quantum,
            "X_outer_train_quantum": Xq_outer_all.iloc[:n_outer_train].reset_index(drop=True),
            "X_outer_test_quantum": Xq_outer_all.iloc[n_outer_train:].reset_index(drop=True),
        })

    HEISENBERG_CACHE_19Q_36F = {
        "cache_key": HEISENBERG_CACHE_KEY,
        "config": RUN_LABEL,
        "n_qubits": N_QUBITS,
        "n_input_features": N_SHAP_FEATURES,
        "n_blocks": HEISENBERG_PQFM_19Q.theta_info["num_blocks"],
        "n_quantum_features": len(quantum_columns),
        "folds": heisenberg_cached_folds,
    }
    print(
        f"Cache concluído: {len(heisenberg_cached_folds)} outer folds | "
        f"{len(quantum_columns)} features quânticas por amostra."
    )


Preparando uma única vez o circuito Heisenberg na GPU...
Circuito congelado: 19 qubits | 18 arestas/bloco | 2 blocos | GPU.
[Heisenberg 01/40] outer=1/10 | inner=1/3 | amostras=50 | GPU
  concluído em 9.8s
[Heisenberg 02/40] outer=1/10 | inner=2/3 | amostras=75 | GPU
  concluído em 15.1s
[Heisenberg 03/40] outer=1/10 | inner=3/3 | amostras=100 | GPU
  concluído em 21.7s
[Heisenberg 04/40] outer=1/10 | OUTER | amostras=190 | GPU
  concluído em 45.0s
[Heisenberg 05/40] outer=2/10 | inner=1/3 | amostras=96 | GPU
  concluído em 27.0s
[Heisenberg 06/40] outer=2/10 | inner=2/3 | amostras=143 | GPU
  concluído em 38.0s
[Heisenberg 07/40] outer=2/10 | inner=3/3 | amostras=190 | GPU
  concluído em 50.9s
[Heisenberg 08/40] outer=2/10 | OUTER | amostras=280 | GPU
  concluído em 75.2s
[Heisenberg 09/40] outer=3/10 | inner=1/3 | amostras=140 | GPU
  concluído em 41.2s
[Heisenberg 10/40] outer=3/10 | inner=2/3 | amostras=210 | GPU
  concluído em 58.0s
[Heisenberg 11/40] outer=3/10 | inner=3/3 | amos

In [11]:
# Nested tuning e avaliação: esta célula consome o cache e não chama a PQFM.
if "EXPERIMENTS" not in globals():
    EXPERIMENTS = {}

RUN_LABEL = HEISENBERG_CACHE_19Q_36F["config"]
VARIANT_LABEL = "heisenberg"
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []
checkpoint_dir = Path("outputs/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for heisenberg_fold in HEISENBERG_CACHE_19Q_36F["folds"]:
    cached_fold = heisenberg_fold["classic_fold"]
    outer_fold = cached_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] tuning/classificação outer fold {outer_fold}/10")

    scenario_scores = {
        "quantum_only": {candidate_key(params): [] for params in candidates},
        "hibrida": {candidate_key(params): [] for params in candidates},
    }

    for inner_data, inner_quantum in zip(
        cached_fold["inner_folds"], heisenberg_fold["inner_quantum"], strict=True
    ):
        X_inner_quantum = inner_quantum["X_train_quantum"]
        X_validation_quantum = inner_quantum["X_validation_quantum"]
        X_inner_hybrid = pd.concat(
            [inner_data["X_train"].reset_index(drop=True), X_inner_quantum], axis=1
        )
        X_validation_hybrid = pd.concat(
            [inner_data["X_validation"].reset_index(drop=True), X_validation_quantum], axis=1
        )

        for params in candidates:
            key = candidate_key(params)
            scenario_scores["quantum_only"][key].append(
                fit_and_score_classifier(
                    X_inner_quantum, inner_data["y_train"],
                    X_validation_quantum, inner_data["y_validation"], params,
                )
            )
            scenario_scores["hibrida"][key].append(
                fit_and_score_classifier(
                    X_inner_hybrid, inner_data["y_train"],
                    X_validation_hybrid, inner_data["y_validation"], params,
                )
            )

    best_params_by_scenario = {}
    for scenario in ("quantum_only", "hibrida"):
        mean_scores = {
            key: float(np.mean(values))
            for key, values in scenario_scores[scenario].items()
        }
        best_key = max(mean_scores, key=mean_scores.get)
        best_params_by_scenario[scenario] = json.loads(best_key)
        for key, values in scenario_scores[scenario].items():
            tuning_rows.append({
                "config": RUN_LABEL,
                "outer_fold": outer_fold,
                "variant": VARIANT_LABEL,
                "scenario": scenario,
                "params": key,
                "mean_inner_average_precision": np.mean(values),
                "std_inner_average_precision": np.std(values, ddof=1),
            })

    fold_rows.append({
        "config": RUN_LABEL,
        "variant": VARIANT_LABEL,
        "scenario": "classica",
        "outer_fold": outer_fold,
        "n_train": len(cached_fold["train_idx"]),
        "n_test": len(cached_fold["test_idx"]),
        "test_start_position": int(cached_fold["test_idx"][0]),
        "test_end_position": int(cached_fold["test_idx"][-1]),
        "best_params": json.dumps(cached_fold["best_classic_params"], sort_keys=True),
        **cached_fold["classic_metrics"],
    })

    X_outer_train_quantum = heisenberg_fold["X_outer_train_quantum"]
    X_outer_test_quantum = heisenberg_fold["X_outer_test_quantum"]
    X_outer_train_hybrid = pd.concat(
        [cached_fold["X_outer_train"].reset_index(drop=True), X_outer_train_quantum], axis=1
    )
    X_outer_test_hybrid = pd.concat(
        [cached_fold["X_outer_test"].reset_index(drop=True), X_outer_test_quantum], axis=1
    )

    for scenario, X_train_rep, X_test_rep in (
        ("quantum_only", X_outer_train_quantum, X_outer_test_quantum),
        ("hibrida", X_outer_train_hybrid, X_outer_test_hybrid),
    ):
        best_params = best_params_by_scenario[scenario]
        classifier = GradientBoostingClassifier(
            random_state=RANDOM_STATE, **best_params
        )
        classifier.fit(X_train_rep, cached_fold["y_outer_train"])
        prediction = classifier.predict(X_test_rep)
        probability = classifier.predict_proba(X_test_rep)[:, 1]
        fold_rows.append({
            "config": RUN_LABEL,
            "variant": VARIANT_LABEL,
            "scenario": scenario,
            "outer_fold": outer_fold,
            "n_train": len(cached_fold["train_idx"]),
            "n_test": len(cached_fold["test_idx"]),
            "test_start_position": int(cached_fold["test_idx"][0]),
            "test_end_position": int(cached_fold["test_idx"][-1]),
            "best_params": json.dumps(best_params, sort_keys=True),
            **compute_metrics(cached_fold["y_outer_test"], prediction, probability),
        })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        checkpoint_dir / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl", "wb"
    ) as handle:
        pickle.dump(checkpoint, handle)

EXPERIMENT_HEISENBERG_RESULT = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
EXPERIMENTS[RUN_LABEL] = EXPERIMENT_HEISENBERG_RESULT
print(f"Experimento {RUN_LABEL} concluído. Execute a próxima célula para a análise estatística.")



[19q_2f_heisenberg_gpu] tuning/classificação outer fold 1/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 2/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 3/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 4/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 5/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 6/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 7/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 8/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 9/10

[19q_2f_heisenberg_gpu] tuning/classificação outer fold 10/10
Experimento 19q_2f_heisenberg_gpu concluído. Execute a próxima célula para a análise estatística.


In [12]:
# Análise separada, no formato compacto do notebook original.
df_stats_current = analise_estatistica(
    EXPERIMENT_HEISENBERG_RESULT["fold_results"],
    titulo=f"{EXPERIMENT_HEISENBERG_RESULT['config'].upper()} — 10 OUTER FOLDS",
)
if "STATISTICAL_ANALYSES" not in globals():
    STATISTICAL_ANALYSES = {}
STATISTICAL_ANALYSES[EXPERIMENT_HEISENBERG_RESULT["config"]] = df_stats_current



  19Q_2F_HEISENBERG_GPU — 10 OUTER FOLDS

Nenhuma melhoria estatisticamente significativa encontrada.

────────────────────────────────────────────────────────────────────────────────────────
TABELA COMPLETA POR VARIANTE:

── Variante: HEISENBERG ──

              Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor                 Status
Clássico vs Quantum-Only     ROC-AUC      0.8109      0.7700  -0.0409 [-0.0840, -0.0014]   0.1602     ↓ Piora (não sig.)
     Clássico vs Híbrido     ROC-AUC      0.8109      0.8033  -0.0076  [-0.0341, 0.0153]   0.8457     ↓ Piora (não sig.)
Clássico vs Quantum-Only      PR-AUC      0.8854      0.8724  -0.0129  [-0.0369, 0.0126]   0.1934     ↓ Piora (não sig.)
     Clássico vs Híbrido      PR-AUC      0.8854      0.8855   0.0002  [-0.0146, 0.0143]   0.6953   ↑ Melhora (não sig.)
Clássico vs Quantum-Only    Accuracy      0.7511      0.6989  -0.0522 [-0.0878, -0.0133]   0.0371 ⚠️ PIORA SIGNIFICATIVA
     Clássico vs Híbri

## 11. Heisenberg — 13 qubits × 12 features (1 bloco), GPU

A cadeia de 13 qubits possui exatamente 12 arestas. As 12 melhores features selecionadas por SHAP ocupam um único bloco, sem padding.

Como na camada Heisenberg 19q/36f, o circuito é ajustado uma única vez e as transformações quânticas leakage-safe são armazenadas separadamente para cada representação de fold. O nested tuning apenas consome esse cache.


In [10]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "13q_1f_heisenberg_gpu"
N_QUBITS = 13
N_SHAP_FEATURES = 12
HEISENBERG_R = 2
HEISENBERG_ALPHA = 0.1
HEISENBERG_SHOTS = 4096
USE_TANH_SCALING = True
MEASURE_2LOCAL_DIAGONAL = False
CLASSIC_CACHE = CLASSIC_CACHE_12
# ────────────────────────────────────────────────────────────────────

HEISENBERG_CACHE_KEY_13Q = (
    RUN_LABEL, id(CLASSIC_CACHE), RANDOM_STATE,
    N_QUBITS, N_SHAP_FEATURES, HEISENBERG_R,
    HEISENBERG_ALPHA, HEISENBERG_SHOTS, USE_TANH_SCALING,
    MEASURE_2LOCAL_DIAGONAL,
)

cache_is_current = (
    "HEISENBERG_CACHE_13Q_12F" in globals()
    and HEISENBERG_CACHE_13Q_12F.get("cache_key") == HEISENBERG_CACHE_KEY_13Q
)

if cache_is_current:
    print("Cache Heisenberg 13q × 12f já existe; nenhuma simulação quântica foi repetida.")
else:
    # O fit congela apenas dimensionalidade, circuito e observáveis.
    # Nenhum valor do dataset ou target é usado neste ajuste.
    HEISENBERG_PQFM_13Q = HeisenbergProjectiveQFM(
        name_file=RUN_LABEL,
        seed=RANDOM_STATE,
        ideal=True,
        simulation=True,
        fakebackend=False,
        shots=HEISENBERG_SHOTS,
        q_enc=N_QUBITS,
        R=HEISENBERG_R,
        alpha=HEISENBERG_ALPHA,
        use_tanh_scaling=USE_TANH_SCALING,
        measure_2local_diagonal=MEASURE_2LOCAL_DIAGONAL,
        use_gpu_statevector=True,
        statevector_device="GPU",
        output_root=f"outputs/pqfm/{RUN_LABEL}",
    )

    print("Preparando uma única vez o circuito Heisenberg 13q na GPU...")
    HEISENBERG_PQFM_13Q.fit(np.zeros((1, N_SHAP_FEATURES), dtype=float))
    assert HEISENBERG_PQFM_13Q.theta_info["features_per_block"] == 12
    assert HEISENBERG_PQFM_13Q.theta_info["num_blocks"] == 1
    assert HEISENBERG_PQFM_13Q.theta_info["total_slots"] == 12
    print("Circuito congelado: 13 qubits | 12 arestas | 1 bloco | GPU.")

    heisenberg_cached_folds = []
    transform_counter = 0
    transform_total = len(CLASSIC_CACHE["folds"]) * (INNER_SPLITS + 1)

    for cached_fold in CLASSIC_CACHE["folds"]:
        outer_fold = cached_fold["outer_fold"]
        inner_quantum = []

        for inner_data in cached_fold["inner_folds"]:
            inner_fold = inner_data["inner_fold"]
            n_inner_train = len(inner_data["X_train"])
            X_inner_all = pd.concat(
                [inner_data["X_train"], inner_data["X_validation"]],
                ignore_index=True,
            )

            transform_counter += 1
            print(
                f"[Heisenberg-13q {transform_counter:02d}/{transform_total:02d}] "
                f"outer={outer_fold}/10 | inner={inner_fold}/3 | "
                f"amostras={len(X_inner_all)} | GPU"
            )
            started_at = time.perf_counter()
            Xq_inner_all_values = HEISENBERG_PQFM_13Q.transform(X_inner_all)
            quantum_columns = [
                f"heisenberg_q_{i}" for i in range(Xq_inner_all_values.shape[1])
            ]
            Xq_inner_all = pd.DataFrame(Xq_inner_all_values, columns=quantum_columns)
            inner_quantum.append({
                "inner_fold": inner_fold,
                "X_train_quantum": Xq_inner_all.iloc[:n_inner_train].reset_index(drop=True),
                "X_validation_quantum": Xq_inner_all.iloc[n_inner_train:].reset_index(drop=True),
            })
            print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        n_outer_train = len(cached_fold["X_outer_train"])
        X_outer_all = pd.concat(
            [cached_fold["X_outer_train"], cached_fold["X_outer_test"]],
            ignore_index=True,
        )
        transform_counter += 1
        print(
            f"[Heisenberg-13q {transform_counter:02d}/{transform_total:02d}] "
            f"outer={outer_fold}/10 | OUTER | amostras={len(X_outer_all)} | GPU"
        )
        started_at = time.perf_counter()
        Xq_outer_all_values = HEISENBERG_PQFM_13Q.transform(X_outer_all)
        quantum_columns = [
            f"heisenberg_q_{i}" for i in range(Xq_outer_all_values.shape[1])
        ]
        Xq_outer_all = pd.DataFrame(Xq_outer_all_values, columns=quantum_columns)
        print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        heisenberg_cached_folds.append({
            "classic_fold": cached_fold,
            "inner_quantum": inner_quantum,
            "X_outer_train_quantum": Xq_outer_all.iloc[:n_outer_train].reset_index(drop=True),
            "X_outer_test_quantum": Xq_outer_all.iloc[n_outer_train:].reset_index(drop=True),
        })

    HEISENBERG_CACHE_13Q_12F = {
        "cache_key": HEISENBERG_CACHE_KEY_13Q,
        "config": RUN_LABEL,
        "n_qubits": N_QUBITS,
        "n_input_features": N_SHAP_FEATURES,
        "n_blocks": HEISENBERG_PQFM_13Q.theta_info["num_blocks"],
        "n_quantum_features": len(quantum_columns),
        "folds": heisenberg_cached_folds,
    }
    print(
        f"Cache concluído: {len(heisenberg_cached_folds)} outer folds | "
        f"{len(quantum_columns)} features quânticas por amostra."
    )


Preparando uma única vez o circuito Heisenberg 13q na GPU...
Circuito congelado: 13 qubits | 12 arestas | 1 bloco | GPU.
[Heisenberg-13q 01/40] outer=1/10 | inner=1/3 | amostras=50 | GPU
  concluído em 6.4s
[Heisenberg-13q 02/40] outer=1/10 | inner=2/3 | amostras=75 | GPU
  concluído em 10.4s
[Heisenberg-13q 03/40] outer=1/10 | inner=3/3 | amostras=100 | GPU
  concluído em 15.0s
[Heisenberg-13q 04/40] outer=1/10 | OUTER | amostras=190 | GPU
  concluído em 27.7s
[Heisenberg-13q 05/40] outer=2/10 | inner=1/3 | amostras=96 | GPU
  concluído em 15.3s
[Heisenberg-13q 06/40] outer=2/10 | inner=2/3 | amostras=143 | GPU
  concluído em 23.0s
[Heisenberg-13q 07/40] outer=2/10 | inner=3/3 | amostras=190 | GPU
  concluído em 31.0s
[Heisenberg-13q 08/40] outer=2/10 | OUTER | amostras=280 | GPU
  concluído em 44.6s
[Heisenberg-13q 09/40] outer=3/10 | inner=1/3 | amostras=140 | GPU
  concluído em 24.8s
[Heisenberg-13q 10/40] outer=3/10 | inner=2/3 | amostras=210 | GPU
  concluído em 34.5s
[Heisenberg

In [11]:
# Nested tuning e avaliação: esta célula consome o cache e não chama a PQFM.
if "EXPERIMENTS" not in globals():
    EXPERIMENTS = {}

RUN_LABEL = HEISENBERG_CACHE_13Q_12F["config"]
VARIANT_LABEL = "heisenberg"
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []
checkpoint_dir = Path("outputs/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for heisenberg_fold in HEISENBERG_CACHE_13Q_12F["folds"]:
    cached_fold = heisenberg_fold["classic_fold"]
    outer_fold = cached_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] tuning/classificação outer fold {outer_fold}/10")

    scenario_scores = {
        "quantum_only": {candidate_key(params): [] for params in candidates},
        "hibrida": {candidate_key(params): [] for params in candidates},
    }

    for inner_data, inner_quantum in zip(
        cached_fold["inner_folds"], heisenberg_fold["inner_quantum"], strict=True
    ):
        X_inner_quantum = inner_quantum["X_train_quantum"]
        X_validation_quantum = inner_quantum["X_validation_quantum"]
        X_inner_hybrid = pd.concat(
            [inner_data["X_train"].reset_index(drop=True), X_inner_quantum], axis=1
        )
        X_validation_hybrid = pd.concat(
            [inner_data["X_validation"].reset_index(drop=True), X_validation_quantum], axis=1
        )

        for params in candidates:
            key = candidate_key(params)
            scenario_scores["quantum_only"][key].append(
                fit_and_score_classifier(
                    X_inner_quantum, inner_data["y_train"],
                    X_validation_quantum, inner_data["y_validation"], params,
                )
            )
            scenario_scores["hibrida"][key].append(
                fit_and_score_classifier(
                    X_inner_hybrid, inner_data["y_train"],
                    X_validation_hybrid, inner_data["y_validation"], params,
                )
            )

    best_params_by_scenario = {}
    for scenario in ("quantum_only", "hibrida"):
        mean_scores = {
            key: float(np.mean(values))
            for key, values in scenario_scores[scenario].items()
        }
        best_key = max(mean_scores, key=mean_scores.get)
        best_params_by_scenario[scenario] = json.loads(best_key)
        for key, values in scenario_scores[scenario].items():
            tuning_rows.append({
                "config": RUN_LABEL,
                "outer_fold": outer_fold,
                "variant": VARIANT_LABEL,
                "scenario": scenario,
                "params": key,
                "mean_inner_average_precision": np.mean(values),
                "std_inner_average_precision": np.std(values, ddof=1),
            })

    fold_rows.append({
        "config": RUN_LABEL,
        "variant": VARIANT_LABEL,
        "scenario": "classica",
        "outer_fold": outer_fold,
        "n_train": len(cached_fold["train_idx"]),
        "n_test": len(cached_fold["test_idx"]),
        "test_start_position": int(cached_fold["test_idx"][0]),
        "test_end_position": int(cached_fold["test_idx"][-1]),
        "best_params": json.dumps(cached_fold["best_classic_params"], sort_keys=True),
        **cached_fold["classic_metrics"],
    })

    X_outer_train_quantum = heisenberg_fold["X_outer_train_quantum"]
    X_outer_test_quantum = heisenberg_fold["X_outer_test_quantum"]
    X_outer_train_hybrid = pd.concat(
        [cached_fold["X_outer_train"].reset_index(drop=True), X_outer_train_quantum], axis=1
    )
    X_outer_test_hybrid = pd.concat(
        [cached_fold["X_outer_test"].reset_index(drop=True), X_outer_test_quantum], axis=1
    )

    for scenario, X_train_rep, X_test_rep in (
        ("quantum_only", X_outer_train_quantum, X_outer_test_quantum),
        ("hibrida", X_outer_train_hybrid, X_outer_test_hybrid),
    ):
        best_params = best_params_by_scenario[scenario]
        classifier = GradientBoostingClassifier(
            random_state=RANDOM_STATE, **best_params
        )
        classifier.fit(X_train_rep, cached_fold["y_outer_train"])
        prediction = classifier.predict(X_test_rep)
        probability = classifier.predict_proba(X_test_rep)[:, 1]
        fold_rows.append({
            "config": RUN_LABEL,
            "variant": VARIANT_LABEL,
            "scenario": scenario,
            "outer_fold": outer_fold,
            "n_train": len(cached_fold["train_idx"]),
            "n_test": len(cached_fold["test_idx"]),
            "test_start_position": int(cached_fold["test_idx"][0]),
            "test_end_position": int(cached_fold["test_idx"][-1]),
            "best_params": json.dumps(best_params, sort_keys=True),
            **compute_metrics(cached_fold["y_outer_test"], prediction, probability),
        })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        checkpoint_dir / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl", "wb"
    ) as handle:
        pickle.dump(checkpoint, handle)

EXPERIMENT_HEISENBERG_13Q_RESULT = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
EXPERIMENTS[RUN_LABEL] = EXPERIMENT_HEISENBERG_13Q_RESULT
print(f"Experimento {RUN_LABEL} concluído. Execute a próxima célula para a análise estatística.")



[13q_1f_heisenberg_gpu] tuning/classificação outer fold 1/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 2/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 3/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 4/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 5/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 6/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 7/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 8/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 9/10

[13q_1f_heisenberg_gpu] tuning/classificação outer fold 10/10
Experimento 13q_1f_heisenberg_gpu concluído. Execute a próxima célula para a análise estatística.


In [12]:
# Análise separada, no formato compacto do notebook original.
df_stats_current = analise_estatistica(
    EXPERIMENT_HEISENBERG_13Q_RESULT["fold_results"],
    titulo=f"{EXPERIMENT_HEISENBERG_13Q_RESULT['config'].upper()} — 10 OUTER FOLDS",
)
if "STATISTICAL_ANALYSES" not in globals():
    STATISTICAL_ANALYSES = {}
STATISTICAL_ANALYSES[EXPERIMENT_HEISENBERG_13Q_RESULT["config"]] = df_stats_current



  13Q_1F_HEISENBERG_GPU — 10 OUTER FOLDS

Nenhuma melhoria estatisticamente significativa encontrada.

────────────────────────────────────────────────────────────────────────────────────────
TABELA COMPLETA POR VARIANTE:

── Variante: HEISENBERG ──

              Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor               Status
Clássico vs Quantum-Only     ROC-AUC      0.8173      0.8138  -0.0035  [-0.0247, 0.0154]   1.0000   ↓ Piora (não sig.)
     Clássico vs Híbrido     ROC-AUC      0.8173      0.8105  -0.0068  [-0.0257, 0.0094]   0.6250   ↓ Piora (não sig.)
Clássico vs Quantum-Only      PR-AUC      0.8883      0.8915   0.0032  [-0.0038, 0.0110]   0.4316 ↑ Melhora (não sig.)
     Clássico vs Híbrido      PR-AUC      0.8883      0.8891   0.0008  [-0.0071, 0.0103]   0.6953 ↑ Melhora (não sig.)
Clássico vs Quantum-Only    Accuracy      0.7500      0.7489  -0.0011  [-0.0167, 0.0167]   1.0000   ↓ Piora (não sig.)
     Clássico vs Híbrido    Accura

## 12. Comparação conjunta opcional

O registro `EXPERIMENTS` contém apenas as células quânticas efetivamente executadas.


In [11]:
if not EXPERIMENTS:
    print("Execute ao menos uma seção PQFM.")
else:
    all_fold_results = pd.concat(
        [experiment["fold_results"] for experiment in EXPERIMENTS.values()],
        ignore_index=True,
    )
    all_summary = summarize_results(all_fold_results)
    all_wilcoxon = paired_wilcoxon_table(all_fold_results)
    display(all_fold_results)
    display(all_summary)
    display(all_wilcoxon)


,config,variant,scenario,outer_fold,n_train,n_test,test_start_position,test_end_position,best_params,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1,Specificity
0,12q_1f,diagonal,classica,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.974085,0.997575,0.944444,0.975309,0.963415,0.969325,0.750000
1,12q_1f,diagonal,quantum_only,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.927591,0.992247,0.855556,0.985915,0.853659,0.915033,0.875000
2,12q_1f,diagonal,hibrida,1,100,90,100,189,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.951220,0.994998,0.844444,0.972222,0.853659,0.909091,0.750000
3,12q_1f,cross-only,classica,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.974085,0.997575,0.944444,0.975309,0.963415,0.969325,0.750000
4,12q_1f,cross-only,quantum_only,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.903201,0.986398,0.944444,0.987342,0.951220,0.968944,0.875000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,12q_1f,cross-only,quantum_only,10,910,90,910,999,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.804409,0.916398,0.733333,0.824561,0.770492,0.796610,0.655172
86,12q_1f,cross-only,hibrida,10,910,90,910,999,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.859243,0.933889,0.744444,0.806452,0.819672,0.813008,0.586207
87,12q_1f,full,classica,10,910,90,910,999,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.822781,0.917903,0.733333,0.824561,0.770492,0.796610,0.655172
88,12q_1f,full,quantum_only,10,910,90,910,999,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.820803,0.919373,0.755556,0.854545,0.770492,0.810345,0.724138


,config,variant,scenario,metric,n_folds,mean,std,ci95_low,ci95_high
0,12q_1f,diagonal,classica,ROC-AUC,10,0.817293,0.095939,0.748662,0.885923
1,12q_1f,diagonal,quantum_only,ROC-AUC,10,0.793042,0.105215,0.717776,0.868308
2,12q_1f,diagonal,hibrida,ROC-AUC,10,0.803684,0.110846,0.724389,0.882979
3,12q_1f,cross-only,classica,ROC-AUC,10,0.817293,0.095939,0.748662,0.885923
4,12q_1f,cross-only,quantum_only,ROC-AUC,10,0.771833,0.107740,0.694761,0.848906
...,...,...,...,...,...,...,...,...,...
58,12q_1f,cross-only,quantum_only,Specificity,10,0.664240,0.130605,0.570810,0.757669
59,12q_1f,cross-only,hibrida,Specificity,10,0.656788,0.131802,0.562503,0.751074
60,12q_1f,full,classica,Specificity,10,0.690129,0.086768,0.628058,0.752199
61,12q_1f,full,quantum_only,Specificity,10,0.684805,0.164774,0.566933,0.802677


,config,variant,metric,baseline,challenger,n_pairs,mean_delta,wilcoxon_statistic,p_value
0,12q_1f,diagonal,ROC-AUC,classica,quantum_only,10,-0.024251,11.0,0.105469
1,12q_1f,diagonal,ROC-AUC,classica,hibrida,10,-0.013609,16.0,0.275391
2,12q_1f,diagonal,ROC-AUC,quantum_only,hibrida,10,0.010642,17.0,0.322266
3,12q_1f,cross-only,ROC-AUC,classica,quantum_only,10,-0.045459,5.0,0.019531
4,12q_1f,cross-only,ROC-AUC,classica,hibrida,10,-0.004687,23.0,0.695312
...,...,...,...,...,...,...,...,...,...
58,12q_1f,cross-only,Specificity,classica,hibrida,10,-0.033340,13.0,0.300781
59,12q_1f,cross-only,Specificity,quantum_only,hibrida,10,-0.007452,14.0,0.640625
60,12q_1f,full,Specificity,classica,quantum_only,10,-0.005324,27.0,1.000000
61,12q_1f,full,Specificity,classica,hibrida,10,-0.009235,21.0,0.910156
